# Primordial Soup: replaying the tape of life, at scale

Runs the evolution simulation headless for many seeds under each setting below, then compares how often each
stage emerges (replicators → cells → proteins → DNA → photosynthesis → Great Oxidation → eukaryotes → sex →
multicellular → complex organisms), when it happens, and what the first cells' genomes looked like.

Simulation code: commit `b7d3d6f`, built 2026-10-15.

Works on any Jupyter server (Linux or macOS) and on Kaggle. **Before running:** if there is no Node.js 18+, the setup
cell downloads it from nodejs.org, and it installs pandas / scipy / matplotlib if they are missing, so it needs internet
access the first time (on Kaggle: switch *Internet* on). It uses the CPUs and memory your container is actually allowed.
Results are saved after every world (`primordial-soup-runs/results-<RUN_TAG>.jsonl`), so if the kernel stops, run all
cells again and it carries on where it left off. Change `RUN_TAG` to start a fresh experiment.


In [ ]:
# ---- Experiment settings -------------------------------------------------------------------------
# Each entry is a set of overrides for js/config.js (CFG). The first one is the baseline.
CONFIGS = {
    # The cap is back at 6,000 (12,000 made late-game ticks ~45 ms in the browser). When crowded, a safety valve
    # culls cells; it now removes the weakest first (drained energy reserve, near old age, or infected) at the same
    # overall rate (CULL_SELECTIVE). Tested both ways, with the nutrient settings both ways (the nutrient cuts kept
    # worlds under the cap but starved eukaryotes in euk2).
    "random_old_nut": {"CULL_SELECTIVE": False, "VENT_NUT": 0.1, "WEATHERING": 0.16, "NUT_RUST": 1},
    "select_old_nut": {"VENT_NUT": 0.1, "WEATHERING": 0.16, "NUT_RUST": 1},
    "random":         {"CULL_SELECTIVE": False},
    "select":         {},                               # today's defaults
}
TICKS = 80_000            # simulated ticks per world (late stages typically arrive at 40k-70k)
SEEDS_PER_CONFIG = 28     # None: chosen from the number of cores (one full wave)
FIRST_SEED = 9001          # use a fresh range for each new experiment: runs are deterministic, so reused seeds
                           # replay the same worlds (seeds 1-54 were used by pool4, sun2 and o2a)
WORKERS = None            # None: as many as the cores and memory allow
RUN_TAG = "cull1"         # results go to primordial-soup-runs/results-<RUN_TAG>.jsonl


In [ ]:
# ---- Environment: cores, memory, Python packages, Node.js -----------------------------------------
import os, sys, json, shutil, subprocess, platform, tarfile, urllib.request, base64, time, importlib.util

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.join(os.getcwd(), "primordial-soup-runs")
os.makedirs(WORK, exist_ok=True)
SIM = os.path.join(WORK, "primordial-soup")

missing = [m for m in ("pandas", "scipy", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    print("Installing", *missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def read(path):
    try:
        with open(path) as f:
            return f.read().strip()
    except OSError:
        return None

def usable_cores():
    # The CPUs this process may run on, and any container (cgroup) quota, which os.cpu_count() ignores.
    n = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    q = read("/sys/fs/cgroup/cpu.max")                          # cgroup v2: "<quota> <period>" or "max ..."
    if q and not q.startswith("max"):
        quota, period = map(int, q.split()[:2]); n = min(n, max(1, quota // period))
    q, p = read("/sys/fs/cgroup/cpu/cpu.cfs_quota_us"), read("/sys/fs/cgroup/cpu/cpu.cfs_period_us")  # v1
    if q and p and int(q) > 0:
        n = min(n, max(1, int(q) // int(p)))
    return n

def free_gb():
    # Memory still available, within the container's limit if it has one.
    avail = None
    for line in (read("/proc/meminfo") or "").splitlines():
        if line.startswith("MemAvailable:"):
            avail = int(line.split()[1]) / 1e6
    if avail is None:
        try:
            import psutil; avail = psutil.virtual_memory().available / 1e9
        except ImportError:
            avail = 8.0
    for lim, cur in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                     ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        L, C = read(lim), read(cur)
        if L and C and L.isdigit() and int(L) < 1 << 60:
            avail = min(avail, (int(L) - int(C)) / 1e9)
    return avail

cores, mem = usable_cores(), free_gb()
GB_PER_WORLD = 0.7        # a late-game world with ~6,000 cells peaks around this
if WORKERS is None:
    WORKERS = max(1, min(cores, int(mem / GB_PER_WORLD)))
if SEEDS_PER_CONFIG is None:
    SEEDS_PER_CONFIG = max(16, WORKERS // len(CONFIGS))
print(f"{cores} usable cores, {mem:.0f} GB free -> {WORKERS} worlds at a time; "
      f"{SEEDS_PER_CONFIG} seeds x {len(CONFIGS)} settings = {SEEDS_PER_CONFIG * len(CONFIGS)} worlds")

def node_version(exe):
    try:
        v = subprocess.run([exe, "--version"], capture_output=True, text=True, timeout=30).stdout.strip()
        return int(v.lstrip("v").split(".")[0]), v
    except Exception:
        return 0, None

NODE = shutil.which("node")
major, ver = node_version(NODE) if NODE else (0, None)
if major < 18:
    arch = {"x86_64": "x64", "amd64": "x64", "aarch64": "arm64", "arm64": "arm64"}.get(platform.machine().lower(), "x64")
    osname = {"linux": "linux", "darwin": "darwin"}.get(sys.platform)
    assert osname, "Install Node.js 18+ yourself on this system (automatic download covers Linux and macOS)"
    name = f"node-v20.18.0-{osname}-{arch}"
    dest = os.path.join(WORK, name)
    if not os.path.isdir(dest):
        ext = "tar.xz" if osname == "linux" else "tar.gz"
        url = f"https://nodejs.org/dist/v20.18.0/{name}.{ext}"
        print("Downloading", url)
        tar = os.path.join(WORK, f"{name}.{ext}")
        urllib.request.urlretrieve(url, tar)
        with tarfile.open(tar) as t:
            t.extractall(WORK)
        os.remove(tar)
    NODE = os.path.join(dest, "bin", "node")
    major, ver = node_version(NODE)
print("Node", ver, "at", NODE)
assert major >= 18, "Node.js 18+ is needed (allow internet access so it can be downloaded, or install it)"


In [ ]:
# ---- The simulation (embedded: its 13 source files, base64) ------------------------------------
FILES = {
 "check-evolution.mjs": "Ly8gIlJlcGxheSB0aGUgdGFwZSBvZiBsaWZlIjogcnVuIG1hbnkgc2VlZHMgaW4gcGFyYWxsZWwgYW5kIHJlcG9ydCB3aGVuIGVhY2ggbWlsZXN0b25lIGhhcHBlbmVkLgovLyBEb3VibGVzIGFzIGEgcmVncmVzc2lvbiBjaGVjayB0aGF0IGV2ZXJ5IGV2b2x1dGlvbmFyeSBzdGFnZSBzdGlsbCBlbWVyZ2VzIGFmdGVyIGEgY2hhbmdlLgovLwovLyAgIG5vZGUgY2hlY2stZXZvbHV0aW9uLm1qcyBbLS1zZWVkcyAxLThdIFstLXRpY2tzIDYwMDAwXSBbLS1qb2JzIDZdIFstLWNmZyAneyJQUkVEX1AiOjAuNH0nXSBbLS1tZCByZXBvcnQubWRdCi8vICAgICAgICAgICAgICAgICAgICAgICAgICAgIFstLW91dCByZXN1bHRzLmpzb25sXSAgIChlYWNoIHdvcmxkJ3MgZnVsbCByZXN1bHQsIHNhdmVkIGFzIHNvb24gYXMgaXQgZmluaXNoZXMpCmltcG9ydCB7IHNwYXduIH0gZnJvbSAnbm9kZTpjaGlsZF9wcm9jZXNzJzsKaW1wb3J0IG9zIGZyb20gJ25vZGU6b3MnOwppbXBvcnQgZnMgZnJvbSAnbm9kZTpmcyc7CmltcG9ydCB7IGZpbGVVUkxUb1BhdGggfSBmcm9tICdub2RlOnVybCc7Cgpjb25zdCBhcmdzID0gT2JqZWN0LmZyb21FbnRyaWVzKHByb2Nlc3MuYXJndi5zbGljZSgyKS5yZWR1Y2UoKGEsIHYsIGksIGFycikgPT4gKHYuc3RhcnRzV2l0aCgnLS0nKSA/IFsuLi5hLCBbdi5zbGljZSgyKSwgYXJyW2kgKyAxXV1dIDogYSksIFtdKSk7CgovLyBTdGFnZXMgd2UgZXhwZWN0LCBpbiByb3VnaCBoaXN0b3JpY2FsIG9yZGVyLCB3aXRoIHRoZSBzaGFyZSBvZiBzZWVkcyB0aGF0IHNob3VsZCByZWFjaCBlYWNoLgpjb25zdCBTVEFHRVMgPSBbCiAgWydyZXBsaWNhdG9yJywgJ1JlcGxpY2F0b3InLCAxLjBdLAogIFsncHJvdG9jZWxsJywgJ1Byb3RvY2VsbCcsIDEuMF0sCiAgWydjZWxsJywgJ0xpdmluZyBjZWxsJywgMS4wXSwKICBbJ3RyYW5zbGF0aW9uJywgJ1Byb3RlaW5zIC8gZ2VuZXRpYyBjb2RlJywgMC45XSwKICBbJ2RuYScsICdETkEgZ2Vub21lJywgMC44XSwKICBbJ3Bob3RvJywgJ1Bob3Rvc3ludGhlc2lzJywgMC44XSwKICBbJ3ZpcnVzJywgJ1ZpcnVzZXMnLCAwLjhdLAogIFsnZ29lJywgJ0dyZWF0IE94aWRhdGlvbicsIDAuNl0sCiAgWydldWthcnlvdGUnLCAnRXVrYXJ5b3RlJywgMC41XSwKICBbJ3NleCcsICdTZXh1YWwgcmVwcm9kdWN0aW9uJywgMC40XSwKICBbJ211bHRpY2VsbHVsYXInLCAnTXVsdGljZWxsdWxhcicsIDAuNF0sCiAgWydjb21wbGV4JywgJ0NvbXBsZXggb3JnYW5pc20nLCAwLjJdLApdOwoKaWYgKGFyZ3Mud29ya2VyICE9PSB1bmRlZmluZWQpIHsKICAvLyBDaGlsZCBwcm9jZXNzOiBzaW11bGF0ZSBvbmUgc2VlZCBhbmQgcHJpbnQgYSBKU09OIHN1bW1hcnkuCiAgY29uc3QgeyBXb3JsZCB9ID0gYXdhaXQgaW1wb3J0KCcuL2pzL3dvcmxkLmpzJyk7CiAgY29uc3QgeyBDRkcgfSA9IGF3YWl0IGltcG9ydCgnLi9qcy9jb25maWcuanMnKTsKICBpZiAoYXJncy5jZmcpIE9iamVjdC5hc3NpZ24oQ0ZHLCBKU09OLnBhcnNlKGFyZ3MuY2ZnKSk7CiAgY29uc3QgR24gPSBhd2FpdCBpbXBvcnQoJy4vanMvZ2Vub21lLmpzJyk7CiAgY29uc3QgdyA9IG5ldyBXb3JsZCgrYXJncy5zZWVkKTsKICBjb25zdCB0aWNrcyA9ICthcmdzLnRpY2tzOwogIC8vIFdhdGNoICh3aXRob3V0IGNoYW5naW5nIGFueXRoaW5nKSB0aGUgZmlyc3QgY2VsbHMgdG8gY29tZSBhbGl2ZTogdGhlaXIgZ2Vub21lcyBhcmUgYnVpbHQgZnJvbQogIC8vIHRoZSBmb3VuZGluZyByZXBsaWNhdG9yJ3Mgc2VxdWVuY2UsIHNvIGVhcmx5IGNoZW1pc3RyeSBzaGFwZXMgZXZlcnkgbGluZWFnZSBhZnRlciBpdC4KICBjb25zdCBmb3VuZGVycyA9IFtdOwogIGNvbnN0IHNwYXduRnJvbSA9IHcubGlmZS5zcGF3bkZyb21Qcm90b2NlbGwuYmluZCh3LmxpZmUpOwogIHcubGlmZS5zcGF3bkZyb21Qcm90b2NlbGwgPSAodiwgcG9seSkgPT4gewogICAgY29uc3QgYyA9IHNwYXduRnJvbSh2LCBwb2x5KTsKICAgIGlmIChmb3VuZGVycy5sZW5ndGggPCAxMCkgZm91bmRlcnMucHVzaCh7IHQ6IHcudCwgbGVuOiBwb2x5LnNlcS5sZW5ndGgsIGFjdDogK3BvbHkuYWN0LnRvRml4ZWQoMyksIGc6IEFycmF5LmZyb20oYy5nLnNsaWNlKDAsIEduLkRJRkYpLCAoeCkgPT4gK3gudG9GaXhlZCg0KSkgfSk7CiAgICByZXR1cm4gYzsKICB9OwogIGNvbnN0IHQwID0gRGF0ZS5ub3coKTsKICBsZXQgcGVha0NlbGxzID0gMDsKICBjb25zdCB0cmFjZSA9IFtdOyAvLyBbdGljaywgY2VsbHMsIGV1a2FyeW90ZXMsIHByZWRhdG9ycywgcGhvdG9zeW50aGVzaXplcnMsIE8yLCBPMiBsb3csIE8yIGhpZ2gsIGFpciBPMiwKICAvLyAgICAgICAgICAgICAgICAgIGV1a2FyeW90aWMgb3JnYW5pc21zLCBsYXJnZXN0LCBtb3N0IHNwZWNpYWxpc2VkIGNlbGwgdHlwZXMsIHdpdGggZ2VybWxpbmUsIGV1a2FyeW90ZSBhZGhlc2lvbl0KICBsZXQgbzJsbyA9IEluZmluaXR5LCBvMmhpID0gLUluZmluaXR5OwogIGNvbnN0IGtleXMgPSBTVEFHRVMubWFwKChba10pID0+IGspOwogIHdoaWxlICh3LnQgPCB0aWNrcykgewogICAgdy5zdGVwKCk7CiAgICAvLyBFdmVyeSBzdGFnZSByZWFjaGVkOiBubyBuZWVkIHRvIGtlZXAgc2ltdWxhdGluZyB0aGlzIHdvcmxkLgogICAgaWYgKHcudCAlIDEwMDAgPT09IDAgJiYga2V5cy5ldmVyeSgoaykgPT4gdy5tc2V0LmhhcyhrKSkpIGJyZWFrOwogICAgaWYgKHcudCAlIDEwMDAgPT09IDApIHBlYWtDZWxscyA9IE1hdGgubWF4KHBlYWtDZWxscywgdy5saWZlLmNlbGxzLmxlbmd0aCk7CiAgICAvLyBBIGNvYXJzZSBoaXN0b3J5IGZvciBkaWFnbm9zaXM6IHBvcHVsYXRpb24sIGV1a2FyeW90ZXMsIHByZWRhdG9ycywgcGhvdG9zeW50aGVzaXplcnMsIG94eWdlbi4KICAgIC8vIE9jZWFuIG94eWdlbiBpcyByZS1tZWFzdXJlZCBldmVyeSAxMDAgdGlja3M6IGtlZXAgaXRzIGxvd2VzdCBhbmQgaGlnaGVzdCBpbiBlYWNoIDIsMDAwLXRpY2sgd2luZG93LgogICAgaWYgKHcudCAlIDEwMCA9PT0gMCkgeyBvMmxvID0gTWF0aC5taW4obzJsbywgdy5vMik7IG8yaGkgPSBNYXRoLm1heChvMmhpLCB3Lm8yKTsgfQogICAgaWYgKHcudCAlIDIwMDAgPT09IDApIHsKICAgICAgY29uc3Qgc3QgPSB3LmxpZmUuc3RhdHMsIEwgPSB3LmxpZmU7CiAgICAgIC8vIEV1a2FyeW90aWMgb3JnYW5pc21zOiBob3cgbWFueSBjb2xvbmllcywgdGhlIGJpZ2dlc3QsIGl0cyBtb3N0IHNwZWNpYWxpc2VkIGNlbGwgdHlwZXMsIGFueSBnZXJtbGluZTsKICAgICAgLy8gYW5kIGhvdyBzdGlja3kgZXVrYXJ5b3RpYyBjZWxscyBhcmUgKGFkaGVzaW9uOiB0aGUgY2hhbmNlIGRhdWdodGVycyBzdGF5IGF0dGFjaGVkKS4KICAgICAgbGV0IGVPcmdzID0gMCwgZUJpZyA9IDAsIGVUeXBlcyA9IDAsIGVHZXJtID0gMCwgYWRoID0gMCwgbkUgPSAwOwogICAgICBmb3IgKGNvbnN0IG8gb2YgTC5vcmdhbmlzbXMpIHsKICAgICAgICBjb25zdCBzcCA9IEwuc3BlY2llcy5nZXQoby5zcGVjaWVzKTsKICAgICAgICBpZiAoIXNwIHx8ICFzcC5ldWsgfHwgby5zaXplIDwgMikgY29udGludWU7CiAgICAgICAgZU9yZ3MrKzsgaWYgKG8uc2l6ZSA+IGVCaWcpIGVCaWcgPSBvLnNpemU7CiAgICAgICAgbGV0IGsgPSAwOyBmb3IgKGxldCB0ID0gMTsgdCA8IDY7IHQrKykgaWYgKG8udHlwZXNbdF0gPiAwKSBrKys7CiAgICAgICAgaWYgKGsgPiBlVHlwZXMpIGVUeXBlcyA9IGs7CiAgICAgICAgaWYgKG8udHlwZXNbNV0gPiAwKSBlR2VybSsrOwogICAgICB9CiAgICAgIGZvciAoY29uc3QgYyBvZiBMLmNlbGxzKSBpZiAoYy5taXRvKSB7IGFkaCArPSBjLmFkaDsgbkUrKzsgfQogICAgICB0cmFjZS5wdXNoKFt3LnQsIEwuY2VsbHMubGVuZ3RoLCBzdC5ldWssIHN0LnByZWQsIHN0LnBob3Rvc3ludGhlc2l6ZXJzLCArdy5vMi50b0ZpeGVkKDMpLCArbzJsby50b0ZpeGVkKDMpLCArbzJoaS50b0ZpeGVkKDMpLCArKHcuYXRtTzIgfHwgMCkudG9GaXhlZCgzKSwKICAgICAgICBlT3JncywgZUJpZywgZVR5cGVzLCBlR2VybSwgbkUgPyArKGFkaCAvIG5FKS50b0ZpeGVkKDMpIDogbnVsbF0pOwogICAgICBvMmxvID0gSW5maW5pdHk7IG8yaGkgPSAtSW5maW5pdHk7CiAgICB9CiAgICBpZiAody50ICUgMjAwMCA9PT0gMCkgcHJvY2Vzcy5zdGRlcnIud3JpdGUoYFBST0dSRVNTICR7dy50fVxuYCk7CiAgfQogIGNvbnN0IHN0ID0gdy5saWZlLnN0YXRzOwogIGNvbnN0IG1zID0gT2JqZWN0LmZyb21FbnRyaWVzKHcubWlsZXN0b25lcy5tYXAoKG0pID0+IFttLmtleSwgbS50XSkpOwogIGNvbnNvbGUubG9nKEpTT04uc3RyaW5naWZ5KHsKICAgIHNlZWQ6ICthcmdzLnNlZWQsIG1zLCBzZWNzOiAoRGF0ZS5ub3coKSAtIHQwKSAvIDEwMDAsIHRpY2tzOiB3LnQsIG5vZGU6IHByb2Nlc3MudmVyc2lvbiwgdHJhY2UsIGRlYXRoczogdy5saWZlLmRlYXRocywgY2ZnOiBhcmdzLmNmZyA/IEpTT04ucGFyc2UoYXJncy5jZmcpIDoge30sIG9yaWdpbnM6IHcubGlmZS5vcmlnaW5zLCBmb3VuZGVycywKICAgIGdlbmVOYW1lczogYXJncy5nZW5lcyA/IEduLkdFTkVfTkFNRVMuc2xpY2UoMCwgR24uRElGRikgOiB1bmRlZmluZWQsCiAgICBmaW5hbDogeyBjZWxsczogdy5saWZlLmNlbGxzLmxlbmd0aCwgcGVha0NlbGxzLCBzcGVjaWVzOiBzdC5zcGVjaWVzQWxpdmUsIGV1azogc3QuZXVrLCBtdWx0aTogc3QubXVsdGljZWxsdWxhciwgbGFyZ2VzdDogc3QubGFyZ2VzdCwgdmlydXNlczogdy52aXJ1c2VzID8gdy52aXJ1c2VzLmxpc3QubGVuZ3RoIDogMCwgbzI6ICt3Lm8yLnRvRml4ZWQoMiksIGVyYTogdy5lcmEoKS5uYW1lIH0sCiAgfSkpOwogIHByb2Nlc3MuZXhpdCgwKTsKfQoKY29uc3QgW2EsIGJdID0gKGFyZ3Muc2VlZHMgfHwgJzEtOCcpLnNwbGl0KCctJykubWFwKE51bWJlcik7CmNvbnN0IHNlZWRzID0gW107CmZvciAobGV0IHMgPSBhOyBzIDw9IChiIHx8IGEpOyBzKyspIHNlZWRzLnB1c2gocyk7CmNvbnN0IHRpY2tzID0gKyhhcmdzLnRpY2tzIHx8IDYwMDAwKTsKY29uc3Qgam9icyA9ICsoYXJncy5qb2JzIHx8IG9zLmNwdXMoKS5sZW5ndGgpOyAvLyBlYWNoIHdvcmxkIGlzIG9uZSBidXN5IGNvcmU7IHVzZSB0aGVtIGFsbApjb25zdCBzZWxmID0gZmlsZVVSTFRvUGF0aChpbXBvcnQubWV0YS51cmwpOwoKY29uc29sZS5sb2coYFJlcGxheWluZyAke3NlZWRzLmxlbmd0aH0gd29ybGRzIMOXICR7dGlja3MudG9Mb2NhbGVTdHJpbmcoKX0gdGlja3Mgb24gJHtqb2JzfSBjb3Jlc+KApmApOwpjb25zdCByZXN1bHRzID0gW107CmNvbnN0IHByb2dyZXNzID0gbmV3IE1hcCgpOwpsZXQgbmV4dCA9IDA7CmNvbnN0IHRTdGFydCA9IERhdGUubm93KCk7CmNvbnN0IHRpY2tlciA9IHNldEludGVydmFsKCgpID0+IHsKICBjb25zdCBkb25lID0gcmVzdWx0cy5sZW5ndGgsIHRvdGFsID0gc2VlZHMubGVuZ3RoOwogIGNvbnN0IHRpY2tzRG9uZSA9IFsuLi5wcm9ncmVzcy52YWx1ZXMoKV0ucmVkdWNlKCh4LCB5KSA9PiB4ICsgeSwgMCkgKyAwOwogIGNvbnN0IGZyYWMgPSBNYXRoLm1pbigxLCB0aWNrc0RvbmUgLyAodG90YWwgKiB0aWNrcykpOwogIGNvbnN0IGVsID0gKERhdGUubm93KCkgLSB0U3RhcnQpIC8gMTAwMDsKICBjb25zdCBldGEgPSBmcmFjID4gMC4wMiA/IGVsIC8gZnJhYyAtIGVsIDogTmFOOwogIGNvbnNvbGUubG9nKGAgIOKApiAkeyhmcmFjICogMTAwKS50b0ZpeGVkKDApfSUgb3ZlcmFsbCAoJHtkb25lfS8ke3RvdGFsfSB3b3JsZHMgZmluaXNoZWQsICR7ZWwudG9GaXhlZCgwKX1zIGVsYXBzZWQke2lzTmFOKGV0YSkgPyAnJyA6IGAsIH4ke01hdGgucm91bmQoZXRhKX1zIGxlZnRgfSlgKTsKfSwgMzAwMDApOwphd2FpdCBuZXcgUHJvbWlzZSgoZG9uZSkgPT4gewogIGxldCBydW5uaW5nID0gMDsKICBjb25zdCBsYXVuY2ggPSAoKSA9PiB7CiAgICB3aGlsZSAocnVubmluZyA8IGpvYnMgJiYgbmV4dCA8IHNlZWRzLmxlbmd0aCkgewogICAgICBjb25zdCBzZWVkID0gc2VlZHNbbmV4dCsrXTsKICAgICAgcnVubmluZysrOwogICAgICBjb25zdCBjcCA9IHNwYXduKHByb2Nlc3MuZXhlY1BhdGgsIFtzZWxmLCAnLS13b3JrZXInLCAnMScsICctLXNlZWQnLCBTdHJpbmcoc2VlZCksICctLXRpY2tzJywgU3RyaW5nKHRpY2tzKSwgLi4uKGFyZ3MuY2ZnID8gWyctLWNmZycsIGFyZ3MuY2ZnXSA6IFtdKV0pOwogICAgICBsZXQgb3V0ID0gJyc7CiAgICAgIGNwLnN0ZG91dC5vbignZGF0YScsIChkKSA9PiAob3V0ICs9IGQpKTsKICAgICAgY3Auc3RkZXJyLm9uKCdkYXRhJywgKGQpID0+IHsKICAgICAgICBmb3IgKGNvbnN0IGxpbmUgb2YgU3RyaW5nKGQpLnNwbGl0KCdcbicpKSB7CiAgICAgICAgICBjb25zdCBtID0gL15QUk9HUkVTUyAoXGQrKS8uZXhlYyhsaW5lKTsKICAgICAgICAgIGlmIChtKSBwcm9ncmVzcy5zZXQoc2VlZCwgK21bMV0pOyBlbHNlIGlmIChsaW5lLnRyaW0oKSkgcHJvY2Vzcy5zdGRlcnIud3JpdGUobGluZSArICdcbicpOwogICAgICAgIH0KICAgICAgfSk7CiAgICAgIGNwLm9uKCdjbG9zZScsICgpID0+IHsKICAgICAgICBydW5uaW5nLS07CiAgICAgICAgcHJvZ3Jlc3Muc2V0KHNlZWQsIHRpY2tzKTsKICAgICAgICB0cnkgeyBjb25zdCByID0gSlNPTi5wYXJzZShvdXQudHJpbSgpLnNwbGl0KCdcbicpLnBvcCgpKTsgcmVzdWx0cy5wdXNoKHIpOyBwcm9jZXNzLnN0ZG91dC53cml0ZShgICBzZWVkICR7c2VlZH0gZG9uZSAoJHtyLnNlY3MudG9GaXhlZCgwKX1zKVxuYCk7IGlmIChhcmdzLm91dCkgZnMuYXBwZW5kRmlsZVN5bmMoYXJncy5vdXQsIEpTT04uc3RyaW5naWZ5KHIpICsgJ1xuJyk7IH0KICAgICAgICBjYXRjaCB7IHJlc3VsdHMucHVzaCh7IHNlZWQsIGVycm9yOiB0cnVlLCBtczoge30sIGZpbmFsOiB7fSB9KTsgY29uc29sZS5sb2coYCAgc2VlZCAke3NlZWR9IEZBSUxFRGApOyB9CiAgICAgICAgaWYgKG5leHQgPj0gc2VlZHMubGVuZ3RoICYmIHJ1bm5pbmcgPT09IDApIGRvbmUoKTsgZWxzZSBsYXVuY2goKTsKICAgICAgfSk7CiAgICB9CiAgfTsKICBsYXVuY2goKTsKfSk7CmNsZWFySW50ZXJ2YWwodGlja2VyKTsKcmVzdWx0cy5zb3J0KCh4LCB5KSA9PiB4LnNlZWQgLSB5LnNlZWQpOwoKLy8gVGFibGU6IG9uZSByb3cgcGVyIHNlZWQsIG9uZSBjb2x1bW4gcGVyIHN0YWdlICh0aWNrIGluIHRob3VzYW5kcykuCmNvbnN0IGZtdCA9ICh0KSA9PiAodCA9PT0gdW5kZWZpbmVkID8gJyAgIOKAkyAgJyA6IGAkeyh0IC8gMTAwMCkudG9GaXhlZCgxKS5wYWRTdGFydCg1KX1rYCk7CmNvbnN0IGhlYWQgPSAnc2VlZCAgJyArIFNUQUdFUy5tYXAoKFssIG5dKSA9PiBuLnNsaWNlKDAsIDYpLnBhZFN0YXJ0KDcpKS5qb2luKCcnKSArICcgICBjZWxscyAgc3AgIGV1ayB2aXJ1ICBlcmEnOwpjb25zdCBsaW5lcyA9IFtoZWFkXTsKZm9yIChjb25zdCByIG9mIHJlc3VsdHMpIHsKICBsaW5lcy5wdXNoKFN0cmluZyhyLnNlZWQpLnBhZFN0YXJ0KDQpICsgJyAgJyArIFNUQUdFUy5tYXAoKFtrXSkgPT4gJyAnICsgZm10KHIubXNba10pKS5qb2luKCcnKSArCiAgICBgICAke1N0cmluZyhyLmZpbmFsLmNlbGxzID8/ICcnKS5wYWRTdGFydCg2KX0gJHtTdHJpbmcoci5maW5hbC5zcGVjaWVzID8/ICcnKS5wYWRTdGFydCgzKX0gJHtTdHJpbmcoci5maW5hbC5ldWsgPz8gJycpLnBhZFN0YXJ0KDQpfSAke1N0cmluZyhyLmZpbmFsLnZpcnVzZXMgPz8gJycpLnBhZFN0YXJ0KDQpfSAgJHtyLmZpbmFsLmVyYSA/PyAnJ31gKTsKfQpjb25zb2xlLmxvZygnXG4nICsgbGluZXMuam9pbignXG4nKSk7Cgpjb25zb2xlLmxvZygnXG5Ib3cgb2Z0ZW4gZWFjaCBzdGFnZSBoYXBwZW5lZCAoYW5kIGl0cyBtZWRpYW4gdGltZSk6Jyk7CmxldCBvayA9IHRydWU7CmNvbnN0IHN1bW1hcnkgPSBbXTsKZm9yIChjb25zdCBbaywgbmFtZSwgd2FudF0gb2YgU1RBR0VTKSB7CiAgY29uc3QgdHMgPSByZXN1bHRzLm1hcCgocikgPT4gci5tc1trXSkuZmlsdGVyKCh0KSA9PiB0ICE9PSB1bmRlZmluZWQpLnNvcnQoKHgsIHkpID0+IHggLSB5KTsKICBjb25zdCBzaGFyZSA9IHRzLmxlbmd0aCAvIHJlc3VsdHMubGVuZ3RoOwogIGNvbnN0IG1lZCA9IHRzLmxlbmd0aCA/IHRzW01hdGguZmxvb3IodHMubGVuZ3RoIC8gMildIDogdW5kZWZpbmVkOwogIGNvbnN0IHBhc3MgPSBzaGFyZSA+PSB3YW50OwogIGlmICghcGFzcykgb2sgPSBmYWxzZTsKICBjb25zdCByb3cgPSBgICAke3Bhc3MgPyAn4pyTJyA6ICfinJcnfSAke25hbWUucGFkRW5kKDI0KX0gJHtTdHJpbmcoTWF0aC5yb3VuZChzaGFyZSAqIDEwMCkpLnBhZFN0YXJ0KDMpfSUgICh3YW50IOKJpSAke01hdGgucm91bmQod2FudCAqIDEwMCl9JSkgICBtZWRpYW4gJHttZWQgPT09IHVuZGVmaW5lZCA/ICfigJQnIDogbWVkLnRvTG9jYWxlU3RyaW5nKCkgKyAnIHRpY2tzJ31gOwogIGNvbnNvbGUubG9nKHJvdyk7CiAgc3VtbWFyeS5wdXNoKHsgbmFtZSwgc2hhcmUsIG1lZCwgd2FudCwgcGFzcyB9KTsKfQpjb25zb2xlLmxvZyhvayA/ICdcbkFsbCBzdGFnZXMgZW1lcmdlIGFzIGV4cGVjdGVkLicgOiAnXG5Tb21lIHN0YWdlcyBlbWVyZ2UgbGVzcyBvZnRlbiB0aGFuIGV4cGVjdGVkIChzZWUg4pyXKS4nKTsKCmlmIChhcmdzLm1kKSB7CiAgY29uc3QgbWQgPSBbYCMgUmVwbGF5aW5nIHRoZSB0YXBlIG9mIGxpZmVcbmAsIGAke3Jlc3VsdHMubGVuZ3RofSB3b3JsZHMsICR7dGlja3MudG9Mb2NhbGVTdHJpbmcoKX0gdGlja3MgZWFjaC5cbmAsCiAgICAnfCBTdGFnZSB8IFdvcmxkcyByZWFjaGluZyBpdCB8IE1lZGlhbiB0aWNrIHwnLCAnfCAtLS0gfCAtLS0gfCAtLS0gfCcsCiAgICAuLi5zdW1tYXJ5Lm1hcCgocykgPT4gYHwgJHtzLm5hbWV9IHwgJHtNYXRoLnJvdW5kKHMuc2hhcmUgKiAxMDApfSUgfCAke3MubWVkID09PSB1bmRlZmluZWQgPyAn4oCUJyA6IHMubWVkLnRvTG9jYWxlU3RyaW5nKCl9IHxgKSwKICAgICcnLCAnYGBgJywgLi4ubGluZXMsICdgYGAnLCAnJ107CiAgZnMud3JpdGVGaWxlU3luYyhhcmdzLm1kLCBtZC5qb2luKCdcbicpKTsKICBjb25zb2xlLmxvZyhgV3JvdGUgJHthcmdzLm1kfWApOwp9CnByb2Nlc3MuZXhpdChvayA/IDAgOiAxKTsK",
 "js/chemistry.js": "...",
 "js/config.js": "Ly8gQWxsIHR1bmFibGUgY29uc3RhbnRzIG9mIHRoZSBzaW11bGF0aW9uIGluIG9uZSBwbGFjZS4KZXhwb3J0IGNvbnN0IENGRyA9IHsKICAvLyBXb3JsZCB1bml0cyBhcmUgMC41IMK1bTogdGhlIGhhYml0YXQgaXMgMS42IMOXIDAuOSBtbS4KICBXOiAzMjAwLAogIEg6IDE4MDAsCgogIC8vIC0tLS0gRW52aXJvbm1lbnQgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogIEZJRUxEX0NFTEw6IDIwLAogIFZFTlRTOiA3LAogIFZFTlRfQ0hFTTogNC4wLCAgICAgICAgICAvLyByZWR1Y2VkIGNoZW1pY2FscyAoSDIsIEgyUykgZW1pdHRlZCBwZXIgdmVudCBwZXIgdGljawogIFZFTlRfR0xPVzogMC4zLCAgICAgICAgIC8vIGZhaW50IHRoZXJtYWwgbGlnaHQgbmVhciB2ZW50cyAoYW5veHlnZW5pYyBwaG90b3N5bnRoZXNpcyBzdGVwcGluZyBzdG9uZSkKICBDSEVNX0RJRkY6IDAuMjUsIENIRU1fREVDQVk6IDAuMDAxNSwgQ0hFTV9IQUxGOiA0LAogIE9SR19ESUZGOiAwLjE1LCBPUkdfREVDQVk6IDAuMDAwNiwKICBPWFlfRElGRjogMC4zLCBPWFlfREVDQVk6IDAuMDAwNSwgT1hZX1JFQUNUOiAwLjAxLCAvLyAoT1hZX0RFQ0FZIHdhcyAwLjAwMTU6IGEgfjQ2MC10aWNrIGhhbGYtbGlmZSwgdG9vIHNob3J0IGZvciBveHlnZW4gdG8gYnVpbGQgdXApCiAgTlVUX0lOSVQ6IDAuMywgICAgICAgICAgIC8vIGNvbnNlcnZlZCBsaW1pdGluZyBudXRyaWVudCAodGhpbmsgcGhvc3BoYXRlKSBwZXIgZ3JpZCBjZWxsCiAgTlVUX0RJRkY6IDAuMywgTlVUX0JVUklBTDogMC4wMDA0LAogIC8vIE94eWdlbmF0ZWQgd2F0ZXIgcnVzdHMgZGlzc29sdmVkIGlyb24gaW50byBveGlkZXMgdGhhdCBiaW5kIHBob3NwaGF0ZSBhbmQgY2FycnkgaXQgdG8gdGhlIHNlYSBmbG9vcjogbnV0cmllbnQKICAvLyBpcyBidXJpZWQgdXAgdG8gTlVUX1JVU1QgdGltZXMgZmFzdGVyIHdoZXJlIHRoZSB3YXRlciBpcyBmdWxseSBveHlnZW5hdGVkICgxID0gbm8gZWZmZWN0KS4KICBOVVRfUlVTVDogMiwgICAgICAgICAgICAgLy8gKDh4IGtlcHQgd29ybGRzIHVuZGVyIHRoZSBjYXAgYnV0IHN0YXJ2ZWQgZXVrYXJ5b3RlczsgMnggaGFkIHRoZSBtb3N0IGV1a2FyeW90ZXMgaW4gcnVzdDEpCiAgVkVOVF9OVVQ6IDAuMSwgICAgICAgICAgLy8gbWluZXJhbHMgZnJvbSB2ZW50cwogIFdFQVRIRVJJTkc6IDAuMTYsICAgICAgICAvLyBtaW5lcmFscyB3YXNoZWQgaW4gYXQgdGhlIHN1cmZhY2UgKHJpdmVycywgZHVzdCkKICAvLyBBdG1vc3BoZXJlOiBhIHdlbGwtbWl4ZWQgYWlyIHJlc2Vydm9pciBzd2FwcGluZyBveHlnZW4gd2l0aCB0aGUgc3VyZmFjZSB3YXRlciwgd2hpY2ggYnVmZmVycyB0aGUgb2NlYW4KICAvLyB0aHJvdWdoIGRheSBhbmQgbmlnaHQgKG1vc3Qgb2YgRWFydGgncyBmcmVlIG94eWdlbiBpcyBpbiB0aGUgYWlyKS4gQVRNX1NJWkUgaXMgaXRzIGNhcGFjaXR5IGluIHJvd3Mgb2YKICAvLyB3YXRlciAoMCA9IG5vIGF0bW9zcGhlcmUpOyBBVE1fRVhDSEFOR0UgdGhlIHNoYXJlIG9mIHRoZSBkaWZmZXJlbmNlIHN3YXBwZWQgcGVyIHRpY2sgYXQgdGhlIHN1cmZhY2UuCiAgQVRNX1NJWkU6IDgsIEFUTV9FWENIQU5HRTogMC4wNSwgQVRNX0RFQ0FZOiAwLjAwMDA1LAogIE8yX1JFRjogMS41LCAgICAgICAgICAgICAgIC8vIG94eWdlbiBsZXZlbCBjb25zaWRlcmVkICJmdWxseSBveHlnZW5hdGVkIgogIERBWV9MRU5HVEg6IDEwMDAsICAgICAgICAvLyB0aWNrcyBwZXIgZGF5ICh3YXMgNCwwMDA6IG5pZ2h0cyBvdXRsYXN0ZWQgb3h5Z2VuLCBzbyBpdCBjcmFzaGVkIGV2ZXJ5IG5pZ2h0KQogIERBWV9NRUFOOiAwLjY1LCBEQVlfU1dJTkc6IDAuMzUsIC8vIHN1bmxpZ2h0IG92ZXIgYSBkYXk6IERBWV9NRUFOIMKxIERBWV9TV0lORyAobmlnaHQgMzAlLCBub29uIDEwMCUpCiAgTElHSFRfREVQVEg6IDEuNiwgICAgICAgIC8vIGxpZ2h0IGF0dGVudWF0aW9uIHdpdGggZGVwdGgKICBDVVJSRU5UOiAwLjEyLAogIENFTExfRFJJRlQ6IDEuMiwgICAgICAgICAvLyBob3cgc3Ryb25nbHkgY3VycmVudHMgY2FycnkgY2VsbHMgKG1pY3JvYmVzIGFyZSBtb3N0bHkgcGFzc2l2ZSkKICBQTFVNRV9SRUFDSDogMC4zNSwgICAgICAgLy8gdmVudCBwbHVtZXMgcmlzZSB0aGlzIGZyYWN0aW9uIG9mIHRoZSBvY2VhbiBkZXB0aAoKICAvLyAtLS0tIFByZWJpb3RpYyBjaGVtaXN0cnkgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICBIQVNIX0NFTEw6IDE2LAogIE1BWF9NT05PTUVSUzogMzUwMCwKICBWRU5UX01PTk9NRVJfUkFURTogMS4zLCAgLy8gcGVyIHZlbnQgcGVyIHRpY2sKICBTVVJGQUNFX01PTk9NRVJfUkFURTogMC4zNSwgLy8gTWlsbGVy4oCTVXJleSBzdHlsZSBzeW50aGVzaXMgYnkgbGlnaHRuaW5nL1VWIGF0IHRoZSBzdXJmYWNlCiAgTElQSURfRlJBQ1RJT046IDAuMywKICBNT05PTUVSX0RFQ0FZOiAwLjAwMDQsCiAgTU9OT19ESUZGVVNJT046IDEuMSwKICBNQVhfUE9MWU1FUlM6IDgwMCwKICBNQVhfUE9MWV9MRU46IDQwLAogIERJTUVSX1NBTVBMRVM6IDI1MCwKICBESU1FUl9SQVRFOiAwLjI1LAogIFBPTFlfUkFURTogMC4yLAogIEhZRFJPTFlTSVM6IDAuMDAwNCwKICBVVl9EQU1BR0U6IDEuNSwKICBSRVBfUkFURTogMC44LAogIE1JTl9BQ1RJVkVfTEVOOiA2LAogIE1BWF9WRVNJQ0xFUzogMjYwLAogIFZFU19OVUNMRUFURTogOCwKICBWRVNfU1BMSVQ6IDM0LAogIFZFU19QT09MOiA2MCwKICBWRVNfTlRfUkVBQ0g6IDEuNSwgICAgICAgIC8vIGhvdyBmYXIgYmV5b25kIGl0cyBtZW1icmFuZSBhIHByb3RvY2VsbCBkcmF3cyBpbiBudWNsZW90aWRlcyAob25jZSBsaWZlIGlzIGFidW5kYW50KQogIFZFU19OVF9SRUFDSF9FQVJMWTogNCwgICAgLy8gLi4uYmVmb3JlIGxpZmUgKG51bGwgPSBzYW1lIGFzIFZFU19OVF9SRUFDSCk6IGZhZGVzIHRvd2FyZCBpdCBhcyBjZWxscyBtdWx0aXBseQogIFZFU19OVF9GQURFOiA1MCwgICAgICAgICAgLy8gbGl2aW5nIGNlbGxzIGF0IHdoaWNoIHRoZSBlYXJseSBleHRyYSByZWFjaCBoYXMgaGFsdmVkCiAgVkVTX01BWF9QT0xZOiAxNCwKICBWRVNfTElQSURfTE9TUzogMC4wMTIsCiAgVkVTX0JVUlNUOiAwLjAwMDI1LAogIENFTExfR0VOOiAyLCAgICAgICAgICAgICAvLyBwcm90b2NlbGwgZ2VuZXJhdGlvbnMgbmVlZGVkIGJlZm9yZSBpdCBjb3VudHMgYXMgYSBsaXZpbmcgY2VsbAogIENFTExfQUNUOiAwLjIsICAgICAgICAgICAvLyBtaW5pbXVtIHJpYm96eW1lIGFjdGl2aXR5IG9mIHRoZSBwcm90b2NlbGwncyBiZXN0IHJlcGxpY2F0b3IKICBDRUxMX0NPUElFUzogMSwgICAgICAgICAgLy8gY29tcGxldGVkIHNlbGYtY29waWVzIGluIHRoZSBwcm90b2NlbGwgbGluZWFnZSAoaGVyZWRpdHkpCiAgTUFYX09SSUdJTlNfUE9QOiAyMDAsICAgIC8vIHN0b3AgbmV3IGFiaW9nZW5lc2lzIG9uY2UgbGlmZSBpcyB0aGlzIGFidW5kYW50CgogIC8vIC0tLS0gTGlmZSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICBUSFJFQURTOiAwLCAgICAgICAgICAgICAgLy8gdGhyZWFkcyBmb3IgdGhlIHBlci10aWNrIHdvcmsgKDAgPSBhdXRvOiBhbGwgY29yZXMgaW4gYSBicm93c2VyIHBhZ2UsIDEgaW4gTm9kZS93b3JrZXJzKQogIE1BWF9DRUxMUzogNjAwMCwgICAgICAgICAvLyBwb3B1bGF0aW9uIGNhcCwgZm9yIENQVSAoMTIsMDAwIG1hZGUgbGF0ZS1nYW1lIHRpY2tzIH40NSBtcyBpbiB0aGUgYnJvd3NlcikKICAvLyBXaGVuIGNyb3dkZWQgKGFib3ZlIDg1JSBvZiB0aGUgY2FwKSBhIHNhZmV0eSB2YWx2ZSBjdWxscyBjZWxscy4gU2VsZWN0aXZlOiB0aGUgd2Vha2VzdCBnbyBmaXJzdCAoZHJhaW5lZCBlbmVyZ3ksCiAgLy8gbmVhciBvbGQgYWdlLCBvciBpbmZlY3RlZCksIHdpdGggdGhlIG92ZXJhbGwgY3VsbCByYXRlIHVuY2hhbmdlZC4gZmFsc2UgPSBhdCByYW5kb20uCiAgQ1VMTF9TRUxFQ1RJVkU6IHRydWUsCiAgTElGRV9IQVNIOiAyNCwKICBNQVhfUjogMTQsCiAgQ0FQX0s6IDIwLAogIEJBU0VfQ09TVDogMC4wMywKICBNQUNISU5FX0NPU1Q6IDAuMDA2LAogIFBIT1RPX0s6IDAuNDUsCiAgQ0hFTU9fSzogMC4wOCwgQ0hFTU9fRUZGOiAyLjUsCiAgSEVUX0s6IDAuMDgsIEhFVF9FRkY6IDIuMiwKICBTUEVDSUFMSVpFOiAxLjE1LCAgICAgICAgLy8gPjEgbWVhbnMgc3BlY2lhbGlzdHMgYmVhdCBnZW5lcmFsaXN0cwogIEFFUk9fQk9OVVM6IDIuMCwKICBBRVJfQ09TVDogMC4wMDYsCiAgQVJNX0NPU1Q6IDAuMDE1LAogIEFESF9DT1NUOiAwLjAwMiwKICBNT1RfQ09TVDogMC4wMTUsCiAgTU9UX1NQRUVEOiAwLjM1LAogIE8yX1lJRUxEOiAwLjEsCiAgVE9YOiAwLjA1LAogIERJVl9DT1NUOiAwLjA2LAogIE1JTl9ESVZfQUdFOiA0MCwKICAvLyBLaW4gcmVjb2duaXRpb246IHByZWRhdG9ycyBzcGFyZSB0aGVpciBvd24gc3BlY2llcyAobWFueSBwcm90aXN0cyByZWNvZ25pc2Uga2luIGJ5IHN1cmZhY2UgbW9sZWN1bGVzKSB1bmxlc3MKICAvLyBzdGFydmluZyBiZWxvdyB0aGlzIHNoYXJlIG9mIHRoZWlyIGVuZXJneSBjYXBhY2l0eSAoY2FubmliYWxpc20gaXMgbW9zdGx5IGEgc3RhcnZhdGlvbiByZXNwb25zZSkuIDAgPSBvZmYuCiAgS0lOX1NQQVJFOiAwLjIsCiAgLy8gVmlydXNlcyBpbmZlY3Qgb25lIGRvbWFpbjogYmFjdGVyaW9waGFnZXMgY2FuJ3QgaW5mZWN0IGV1a2FyeW90ZXMgKGRpZmZlcmVudCBzdXJmYWNlcywgZW50cnkgYW5kIG1hY2hpbmVyeSkgYW5kCiAgLy8gZXVrYXJ5b3RpYyB2aXJ1c2VzIGNhbid0IGluZmVjdCBiYWN0ZXJpYS4gQSBuZXcgdmlydXMgcGFydGljbGUganVtcHMgdG8gdGhlIG90aGVyIGRvbWFpbiB3aXRoIFZJUlVTX0pVTVAuCiAgVklSVVNfRE9NQUlOOiB0cnVlLCBWSVJVU19KVU1QOiAwLjAwMSwKICBQUkVEX1A6IDAuNCwgICAgICAgICAgICAgLy8gY2hhbmNlIGEgaHVudCBzdWNjZWVkcyAod2FzIDAuMjU6IHByZWRhdG9ycyBzdGF5ZWQgcmFyZSkKICBFTkRPX1A6IDAuMDUsCiAgLy8gRW5kb3N5bWJpb3NpcyBieSBwYXJ0bmVyc2hpcCAoY2xvc2VyIHRvIHRoZSBsZWFkaW5nIGh5cG90aGVzaXM6IGFuIGFyY2hhZWFsIGhvc3QgYW5kIGFuIGFlcm9iaWMgYmFjdGVyaXVtCiAgLy8gbGl2aW5nIGluIGNsb3NlIGNvbnRhY3QgbWVyZ2UsIHdpdGhvdXQgdGhlIGhvc3QgaHVudGluZyk6IGNoYW5jZSBwZXIgY29udGFjdC4gMCA9IG9ubHkgdGhlIHByZWRhdGlvbiByb3V0ZS4KICBTWU1CSU9fUDogMC4wMDA1LAogIEVVS19OVVQ6IDAuNSwgICAgICAgICAgICAvLyBudXRyaWVudCBhIGV1a2FyeW90ZSBuZWVkcyB0byBkaXZpZGUsIHJlbGF0aXZlIHRvIGEgYmFjdGVyaXVtIG9mIGl0cyBzaXplCiAgTUlUT19BRVI6IDAuOSwgICAgICAgICAgIC8vIGFlcm9iaWMgY2FwYWNpdHkgYSBldWthcnlvdGUgZ2V0cyBmcm9tIGl0cyBtaXRvY2hvbmRyaWEgKDAgPSBvbmx5IGl0cyBvd24gZ2VuZSkKICBTWU1CSU9fTzI6IDAuMDUsICAgICAgICAgLy8gbG9jYWwgb3h5Z2VuIChmcmFjdGlvbiBvZiBPMl9SRUYpIHRoZSBwYXJ0bmVyc2hpcCBuZWVkcwogIEhHVF9QOiAwLjAwMDQsCiAgU1BFQ0lBVEU6IDAuMDcsCiAgREFNUDogMC44MiwKICBCT05EX0JSRUFLOiAwLjAwMDE1LAogIFNIQVJFX0s6IDAuMSwKICBNQVhfQk9ORFM6IDUsCiAgLy8gTXVsdGljZWxsdWxhcml0eSBhcyBvbiBFYXJ0aDogYmFjdGVyaWEgZm9ybSBjb2xvbmllcyBhbmQgc2ltcGxlIG11bHRpY2VsbHVsYXIgb3JnYW5pc21zIChubyBtb3V0aCBvcgogIC8vIGdlcm1saW5lIGNlbGxzOiBlbmd1bGZpbmcgbmVlZHMgYSBldWthcnlvdGljIGN5dG9za2VsZXRvbiksIHNvIGNvbXBsZXggb3JnYW5pc21zIGFyZSBldWthcnlvdGljOyBhbmQKICAvLyBib2RpZXMgYWJvdmUgTzJfU0laRV9NSU4gY2VsbHMgbmVlZCBveHlnZW4sIHVwIHRvIE8yX1NJWkVfTUFYIGNlbGxzIHdoZW4gZnVsbHkgb3h5Z2VuYXRlZC4KICBSRUFMSVNUSUNfTVVMVEk6IHRydWUsCiAgTzJfU0laRV9NSU46IDE2LCBPMl9TSVpFX01BWDogNjIsCiAgR0VORVJBTElTVF9CSUFTOiAwLjIsCgogIC8vIE1ham9yIHRyYW5zaXRpb25zCiAgUklCT1NPTUVfQ09TVDogMC4wMDQsICAgIC8vIHVwa2VlcCBvZiBwcm90ZWluLXN5bnRoZXNpcyBtYWNoaW5lcnksIHBlciB1bml0IHRyYW5zbGF0aW9uCiAgUk5BX0dFTk9NRV9DT1NUOiAwLjAwNCwgIC8vIHJlcGFpcmluZyBhIGZyYWdpbGUgUk5BIGdlbm9tZSAow5czIGluIGZ1bGwgb3h5Z2VuKQogIEROQV9DT1NUOiAwLjAwMiwgICAgICAgICAvLyBETkEgZ2Vub21lIHVwa2VlcAogIEROQV9NVVQ6IDAuMzUsICAgICAgICAgICAvLyBtdXRhdGlvbi1yYXRlIG11bHRpcGxpZXIgZm9yIHByb29mcmVhZCBETkEgZ2Vub21lcwogIFNFWF9DT1NUOiAwLjA4LCAgICAgICAgICAvLyBlbmVyZ3kgZWFjaCBwYXJlbnQgc3BlbmRzIG9uIGEgc2V4dWFsIGJpcnRoIChmcmFjdGlvbiBvZiBjYXBhY2l0eSkKCiAgLy8gVmlydXNlcwogIFZJUlVTX09SSUdJTl9QT1A6IDQwMCwgICAvLyBjZWxscyBuZWVkZWQgYmVmb3JlIHNlbGZpc2ggZWxlbWVudHMgY2FuIGVzY2FwZSBhcyB2aXJ1c2VzCiAgVklSVVNfT1JJR0lOX1A6IDAuMDAwNSwKICBWSVJVU19ERUNBWTogMC4wMDMsICAgICAgLy8gcGVyIHRpY2sgKMOXNCBhdCB0aGUgc3VyZmFjZTogVVYpCiAgVklSVVNfRDogMS41LCAgICAgICAgICAgIC8vIEJyb3duaWFuIHN0ZXAgKHVuaXRzL3RpY2spCiAgVklSVVNfSU5GRUNUOiAwLjM1LAogIFZJUlVTX1NQRUM6IDAuMTIsICAgICAgICAvLyBob3cgY2xvc2VseSB0aGUga2V5IG11c3QgbWF0Y2ggdGhlIHJlY2VwdG9yCiAgVklSVVNfTEFURU5DWTogNjAsCiAgVklSVVNfQlVSU1Q6IDEwLAogIFZJUlVTX01VVDogMC4wNCwKICBNQVhfVklSVVNFUzogNDAwMCwKICBUUkFOU0RVQ0U6IDAuMSwKICBSRVNJU1RfQ09TVDogMC4wMDgsICAgIC8vIGhvdyBzdHJvbmdseSBjb2xvbnkgY2VsbHMgZGVmYXVsdCB0byB0aGUgdW5kaWZmZXJlbnRpYXRlZCBmYXRlCiAgREVBVEhfT1JHOiAwLjYsCiAgTlVUX1BFUl9BUkVBOiAyLjAsICAgICAgIC8vIG51dHJpZW50IGxvY2tlZCBpbnRvIGVhY2ggdW5pdCBvZiBjZWxsIGJvZHkKICBOVVRfVVBUQUtFOiAwLjA2LAp9Owo=",
 "js/fields.js": "Ly8gRGlmZnVzaW5nIGNoZW1pY2FsIGNvbmNlbnRyYXRpb24gZmllbGRzIG9uIGEgY29hcnNlIGdyaWQuCmV4cG9ydCBjbGFzcyBGaWVsZHMgewogIGNvbnN0cnVjdG9yKFcsIEgsIGNzKSB7CiAgICB0aGlzLmNzID0gY3M7CiAgICB0aGlzLmd3ID0gTWF0aC5jZWlsKFcgLyBjcyk7CiAgICB0aGlzLmdoID0gTWF0aC5jZWlsKEggLyBjcyk7CiAgICB0aGlzLm4gPSB0aGlzLmd3ICogdGhpcy5naDsKICAgIHRoaXMuY2hlbSA9IG5ldyBGbG9hdDMyQXJyYXkodGhpcy5uKTsgICAgIC8vIHJlZHVjZWQgY2hlbWljYWxzIGZyb20gdmVudHMKICAgIHRoaXMub3JnYW5pYyA9IG5ldyBGbG9hdDMyQXJyYXkodGhpcy5uKTsgIC8vIGRpc3NvbHZlZCBvcmdhbmljIG1hdHRlcgogICAgdGhpcy5veHlnZW4gPSBuZXcgRmxvYXQzMkFycmF5KHRoaXMubik7ICAgLy8gZnJlZSBPMiBmcm9tIHBob3Rvc3ludGhlc2lzCiAgICB0aGlzLm51dHJpZW50ID0gbmV3IEZsb2F0MzJBcnJheSh0aGlzLm4pOyAvLyBsaW1pdGluZyBtaW5lcmFsIG51dHJpZW50LCByZWN5Y2xlZCB0aHJvdWdoIGJvZGllcwogICAgdGhpcy50bXAgPSBuZXcgRmxvYXQzMkFycmF5KHRoaXMubik7CiAgfQoKICBpZHgoeCwgeSkgewogICAgbGV0IGN4ID0gKHggLyB0aGlzLmNzKSB8IDAsIGN5ID0gKHkgLyB0aGlzLmNzKSB8IDA7CiAgICBpZiAoY3ggPCAwKSBjeCA9IDA7IGVsc2UgaWYgKGN4ID49IHRoaXMuZ3cpIGN4ID0gdGhpcy5ndyAtIDE7CiAgICBpZiAoY3kgPCAwKSBjeSA9IDA7IGVsc2UgaWYgKGN5ID49IHRoaXMuZ2gpIGN5ID0gdGhpcy5naCAtIDE7CiAgICByZXR1cm4gY3kgKiB0aGlzLmd3ICsgY3g7CiAgfQoKICBnZXQoZiwgeCwgeSkgeyByZXR1cm4gZlt0aGlzLmlkeCh4LCB5KV07IH0KICBhZGQoZiwgeCwgeSwgYSkgeyBmW3RoaXMuaWR4KHgsIHkpXSArPSBhOyB9CiAgdGFrZShmLCB4LCB5LCBhKSB7CiAgICBjb25zdCBpID0gdGhpcy5pZHgoeCwgeSk7CiAgICBjb25zdCB2ID0gZltpXTsKICAgIGNvbnN0IHQgPSB2IDwgYSA/IHYgOiBhOwogICAgZltpXSA9IHYgLSB0OwogICAgcmV0dXJuIHQ7CiAgfQoKICAvLyBXcml0ZXMgYSByb3VnaGx5IHVuaXQgZ3JhZGllbnQgdmVjdG9yIGludG8gb3V0OyBtYWduaXR1ZGUgc2hyaW5rcyB3aGVuIHRoZSBmaWVsZCBpcyBmbGF0LgogIGdyYWQoZiwgeCwgeSwgb3V0KSB7CiAgICBjb25zdCBndyA9IHRoaXMuZ3csIGdoID0gdGhpcy5naDsKICAgIGxldCBjeCA9ICh4IC8gdGhpcy5jcykgfCAwLCBjeSA9ICh5IC8gdGhpcy5jcykgfCAwOwogICAgaWYgKGN4IDwgMSkgY3ggPSAxOyBlbHNlIGlmIChjeCA+IGd3IC0gMikgY3ggPSBndyAtIDI7CiAgICBpZiAoY3kgPCAxKSBjeSA9IDE7IGVsc2UgaWYgKGN5ID4gZ2ggLSAyKSBjeSA9IGdoIC0gMjsKICAgIGNvbnN0IGkgPSBjeSAqIGd3ICsgY3g7CiAgICBjb25zdCBneCA9IGZbaSArIDFdIC0gZltpIC0gMV07CiAgICBjb25zdCBneSA9IGZbaSArIGd3XSAtIGZbaSAtIGd3XTsKICAgIGNvbnN0IG0gPSBNYXRoLmh5cG90KGd4LCBneSk7CiAgICBpZiAobSA8IDFlLTYpIHsgb3V0LnggPSAwOyBvdXQueSA9IDA7IHJldHVybiBvdXQ7IH0KICAgIGNvbnN0IHJlbCA9IE1hdGgubWluKDEsIChtIC8gKGZbaV0gKyAwLjUpKSAqIDUpOwogICAgb3V0LnggPSAoZ3ggLyBtKSAqIHJlbDsKICAgIG91dC55ID0gKGd5IC8gbSkgKiByZWw7CiAgICByZXR1cm4gb3V0OwogIH0KCiAgZGlmZnVzZShmLCByYXRlLCBkZWNheSkgewogICAgY29uc3QgZ3cgPSB0aGlzLmd3LCBnaCA9IHRoaXMuZ2gsIHQgPSB0aGlzLnRtcDsKICAgIGNvbnN0IGtlZXAgPSAxIC0gZGVjYXk7CiAgICBmb3IgKGxldCB5ID0gMDsgeSA8IGdoOyB5KyspIHsKICAgICAgY29uc3Qgcm93ID0geSAqIGd3OwogICAgICBmb3IgKGxldCB4ID0gMDsgeCA8IGd3OyB4KyspIHsKICAgICAgICBjb25zdCBpID0gcm93ICsgeDsKICAgICAgICBjb25zdCBjID0gZltpXTsKICAgICAgICBjb25zdCBsID0geCA+IDAgPyBmW2kgLSAxXSA6IGM7CiAgICAgICAgY29uc3QgciA9IHggPCBndyAtIDEgPyBmW2kgKyAxXSA6IGM7CiAgICAgICAgY29uc3QgdSA9IHkgPiAwID8gZltpIC0gZ3ddIDogYzsKICAgICAgICBjb25zdCBkID0geSA8IGdoIC0gMSA/IGZbaSArIGd3XSA6IGM7CiAgICAgICAgdFtpXSA9IChjICsgcmF0ZSAqICgobCArIHIgKyB1ICsgZCkgKiAwLjI1IC0gYykpICoga2VlcDsKICAgICAgfQogICAgfQogICAgZi5zZXQodCk7CiAgfQoKICBtZWFuKGYpIHsKICAgIGxldCBzID0gMDsKICAgIGZvciAobGV0IGkgPSAwOyBpIDwgZi5sZW5ndGg7IGkrKykgcyArPSBmW2ldOwogICAgcmV0dXJuIHMgLyBmLmxlbmd0aDsKICB9Cn0K",
 "js/genome.js": "Ly8gRml4ZWQtbG9jdXMgZ2Vub21lLiBUcmFpdCBnZW5lcyBsaXZlIGluIFswLDFdOyBiZWhhdmlvdXIvcmVndWxhdG9yeSB3ZWlnaHRzIGluIFstMSwxXS4KaW1wb3J0IHsgUk5HLCBoYXNoU2VxIH0gZnJvbSAnLi9ybmcuanMnOwoKZXhwb3J0IGNvbnN0IFNJWkUgPSAwLCBQSE9UTyA9IDEsIENIRU1PID0gMiwgSEVURVJPID0gMywgUFJFRCA9IDQsIE1PVElMID0gNSwgQUVST0JJQyA9IDYsCiAgRElWVEggPSA3LCBBREhFUyA9IDgsIFNIQVJFID0gOSwgQVJNT1IgPSAxMCwgTVVUUiA9IDExLCBIVUUgPSAxMiwgTElGRVNQQU4gPSAxMywgQ09MTUFYID0gMTQ7Ci8vIE1ham9yLXRyYW5zaXRpb24gYW5kIGFybXMtcmFjZSBnZW5lcy4KZXhwb3J0IGNvbnN0IFRSQU5TTCA9IDE1OyAgIC8vIHByb3RlaW4gdHJhbnNsYXRpb246IHJpYm9zb21lcyArIGdlbmV0aWMgY29kZSByZXBsYWNlIHJpYm96eW1lIG1ldGFib2xpc20KZXhwb3J0IGNvbnN0IEROQUcgPSAxNjsgICAgIC8vIEROQSBnZW5vbWUgKD4gMC41KTogc3RhYmxlIHN0b3JhZ2UgaW5zdGVhZCBvZiBSTkEKZXhwb3J0IGNvbnN0IFNFWCA9IDE3OyAgICAgIC8vIHRlbmRlbmN5IHRvIHJlcHJvZHVjZSBzZXh1YWxseSAoZXVrYXJ5b3RlcyBvbmx5KQpleHBvcnQgY29uc3QgUkVDMSA9IDE4LCBSRUMyID0gMTk7IC8vIHNoYXBlIG9mIHN1cmZhY2UgcmVjZXB0b3JzOiB3aGF0IHZpcnVzZXMgbGF0Y2ggb250bwpleHBvcnQgY29uc3QgUkVTSVNUID0gMjA7ICAgLy8gQ1JJU1BSLWxpa2UgYW50aXZpcmFsIGRlZmVuY2UKZXhwb3J0IGNvbnN0IFdfTElHSFQgPSAyMSwgV19DSEVNID0gMjIsIFdfT1JHID0gMjMsIFdfUFJFWSA9IDI0LCBXX0ZMRUUgPSAyNSwgV19PWFkgPSAyNjsKZXhwb3J0IGNvbnN0IEZJUlNUX1NJR05FRCA9IFdfTElHSFQ7Ci8vIEdlbmUtcmVndWxhdG9yeSB3ZWlnaHRzIGRlY2lkaW5nIGNlbGwgZmF0ZSBpbnNpZGUgYSBjb2xvbnk6IGZvciBlYWNoIGNlbGwgdHlwZSwKLy8gYSB3ZWlnaHRlZCBzdW0gb2YgW2JpYXMsICNuZWlnaGJvdXJzLCBkZXB0aCBmcm9tIGZvdW5kZXIsIGNvbG9ueSBzaXplLCBsaWdodF0uCmV4cG9ydCBjb25zdCBESUZGID0gMjcsIE5UWVBFUyA9IDYsIE5ESUZGX0lOID0gNTsKZXhwb3J0IGNvbnN0IE5HID0gRElGRiArIE5UWVBFUyAqIE5ESUZGX0lOOwoKZXhwb3J0IGNvbnN0IEdFTkVfTkFNRVMgPSBbCiAgJ1NpemUnLCAnUGhvdG9zeW50aGVzaXMnLCAnQ2hlbW9zeW50aGVzaXMnLCAnSGV0ZXJvdHJvcGh5JywgJ1ByZWRhdGlvbicsICdNb3RpbGl0eScsICdBZXJvYmljIHJlc3AuJywKICAnRGl2aXNpb24gdGhyZXNob2xkJywgJ0FkaGVzaW9uJywgJ0VuZXJneSBzaGFyaW5nJywgJ0FybW9yJywgJ011dGF0aW9uIHJhdGUnLCAnUGlnbWVudCBodWUnLCAnTGlmZXNwYW4nLCAnQ29sb255IHNpemUnLAogICdQcm90ZWluIHRyYW5zbGF0aW9uJywgJ0ROQSBnZW5vbWUnLCAnU2V4dWFsIHJlcHJvZHVjdGlvbicsICdSZWNlcHRvciBzaGFwZSBBJywgJ1JlY2VwdG9yIHNoYXBlIEInLCAnVmlyYWwgZGVmZW5jZScsCiAgJ1NlZWsgbGlnaHQnLCAnU2VlayBjaGVtaWNhbHMnLCAnU2VlayBvcmdhbmljcycsICdDaGFzZSBwcmV5JywgJ0ZsZWUgcHJlZGF0b3JzJywgJ1NlZWsgb3h5Z2VuJywKXTsKCi8vIFRoZSBmaXJzdCBjZWxsJ3MgZ2Vub21lIGlzIGRlcml2ZWQgZnJvbSB0aGUgc2VxdWVuY2Ugb2YgdGhlIHJlcGxpY2F0b3IgdGhhdCBmb3VuZGVkIGl0LgovLyBJdCBpcyBhbiBSTkEtd29ybGQgY2VsbDogcmlib3p5bWVzIGRvIHRoZSBjaGVtaXN0cnkgYW5kIHRoZSBnZW5vbWUgaXMgUk5BLgpleHBvcnQgZnVuY3Rpb24gZnJvbVNlcXVlbmNlKHNlcSkgewogIGNvbnN0IHIgPSBuZXcgUk5HKGhhc2hTZXEoc2VxKSk7CiAgY29uc3QgZyA9IG5ldyBGbG9hdDMyQXJyYXkoTkcpOwogIGdbU0laRV0gPSAwLjI1ICsgci5uZXh0KCkgKiAwLjE1OwogIGdbUEhPVE9dID0gci5uZXh0KCkgKiAwLjA0OwogIGdbQ0hFTU9dID0gMC42ICsgci5uZXh0KCkgKiAwLjI7CiAgZ1tIRVRFUk9dID0gMC4xNSArIHIubmV4dCgpICogMC4yOwogIGdbUFJFRF0gPSByLm5leHQoKSAqIDAuMDM7CiAgZ1tNT1RJTF0gPSAwLjEgKyByLm5leHQoKSAqIDAuMjsKICBnW0FFUk9CSUNdID0gci5uZXh0KCkgKiAwLjAzOwogIGdbRElWVEhdID0gMC40ICsgci5uZXh0KCkgKiAwLjM7CiAgZ1tBREhFU10gPSByLm5leHQoKSAqIDAuMDY7CiAgZ1tTSEFSRV0gPSByLm5leHQoKSAqIDAuMjsKICBnW0FSTU9SXSA9IHIubmV4dCgpICogMC4wNTsKICBnW01VVFJdID0gMC4zICsgci5uZXh0KCkgKiAwLjI7CiAgZ1tIVUVdID0gci5uZXh0KCk7CiAgZ1tMSUZFU1BBTl0gPSAwLjMgKyByLm5leHQoKSAqIDAuMzsKICBnW0NPTE1BWF0gPSByLm5leHQoKSAqIDAuMTsKICBnW1RSQU5TTF0gPSAwLjE1ICsgci5uZXh0KCkgKiAwLjE1OwogIGdbRE5BR10gPSByLm5leHQoKSAqIDAuMTsKICBnW1NFWF0gPSAwLjA1ICsgci5uZXh0KCkgKiAwLjE7IC8vIG9ubHkgZXhwcmVzc2VkIG9uY2UgYSBsaW5lYWdlIGJlY29tZXMgZXVrYXJ5b3RpYwogIGdbUkVDMV0gPSByLm5leHQoKTsKICBnW1JFQzJdID0gci5uZXh0KCk7CiAgZ1tSRVNJU1RdID0gci5uZXh0KCkgKiAwLjA1OwogIGdbV19MSUdIVF0gPSAoci5uZXh0KCkgLSAwLjUpICogMC40OwogIGdbV19DSEVNXSA9IDAuNCArIHIubmV4dCgpICogMC4zOwogIGdbV19PUkddID0gci5uZXh0KCkgKiAwLjI7CiAgZ1tXX1BSRVldID0gci5uZXh0KCkgKiAwLjM7CiAgZ1tXX0ZMRUVdID0gci5uZXh0KCkgKiAwLjM7CiAgZ1tXX09YWV0gPSAtci5uZXh0KCkgKiAwLjI7CiAgZm9yIChsZXQgaSA9IERJRkY7IGkgPCBORzsgaSsrKSBnW2ldID0gKHIubmV4dCgpIC0gMC41KSAqIDAuMzsKICByZXR1cm4gZzsKfQoKLy8gcmF0ZU11bCA8IDEgZm9yIEROQSBnZW5vbWVzIChwcm9vZnJlYWRpbmcgYW5kIHJlcGFpcikuCmV4cG9ydCBmdW5jdGlvbiBtdXRhdGUocCwgcm5nLCByYXRlTXVsID0gMSkgewogIGNvbnN0IGcgPSBuZXcgRmxvYXQzMkFycmF5KHApOwogIGNvbnN0IHJhdGUgPSAoMC4wMSArIHBbTVVUUl0gKiAwLjA1KSAqIHJhdGVNdWw7IC8vIHRoZSBtdXRhdGlvbiByYXRlIGl0c2VsZiBldm9sdmVzCiAgZm9yIChsZXQgaSA9IDA7IGkgPCBORzsgaSsrKSB7CiAgICBpZiAoaSA9PT0gSFVFIHx8IHJuZy5uZXh0KCkgPj0gcmF0ZSkgY29udGludWU7CiAgICBpZiAocm5nLm5leHQoKSA8IDAuMDQpIGdbaV0gPSBpID49IEZJUlNUX1NJR05FRCA/IHJuZy5yYW5nZSgtMSwgMSkgOiBybmcubmV4dCgpOyAvLyBsYXJnZS1lZmZlY3QgbXV0YXRpb24KICAgIGVsc2UgZ1tpXSArPSBybmcuZ2F1c3MoKSAqIDAuMTsKICAgIGNvbnN0IGxvID0gaSA+PSBGSVJTVF9TSUdORUQgPyAtMSA6IDA7CiAgICBpZiAoZ1tpXSA8IGxvKSBnW2ldID0gbG87IGVsc2UgaWYgKGdbaV0gPiAxKSBnW2ldID0gMTsKICB9CiAgLy8gTmV1dHJhbCBwaWdtZW50IGRyaWZ0cyBldmVyeSBnZW5lcmF0aW9uOiBhIHZpc2libGUgbW9sZWN1bGFyIGNsb2NrLgogIGdbSFVFXSA9IChnW0hVRV0gKyBybmcuZ2F1c3MoKSAqIDAuMDA0ICsgMSkgJSAxOwogIHJldHVybiBnOwp9CgovLyBTZXh1YWwgcmVwcm9kdWN0aW9uOiBlYWNoIGdlbmUgY29tZXMgZnJvbSBvbmUgcGFyZW50IG9yIHRoZSBvdGhlciAoZnJlZSByZWNvbWJpbmF0aW9uKS4KZXhwb3J0IGZ1bmN0aW9uIHJlY29tYmluZShhLCBiLCBybmcpIHsKICBjb25zdCBnID0gbmV3IEZsb2F0MzJBcnJheShhLmxlbmd0aCk7CiAgZm9yIChsZXQgaSA9IDA7IGkgPCBhLmxlbmd0aDsgaSsrKSBnW2ldID0gcm5nLm5leHQoKSA8IDAuNSA/IGFbaV0gOiBiW2ldOwogIHJldHVybiBnOwp9CgpleHBvcnQgZnVuY3Rpb24gZGlzdGFuY2UoYSwgYikgewogIGxldCBzID0gMDsKICBmb3IgKGxldCBpID0gMDsgaSA8IE5HOyBpKyspIHsKICAgIGlmIChpID09PSBIVUUpIGNvbnRpbnVlOwogICAgY29uc3QgZCA9IE1hdGguYWJzKGFbaV0gLSBiW2ldKTsKICAgIHMgKz0gaSA+PSBGSVJTVF9TSUdORUQgPyBkICogMC41IDogZDsKICB9CiAgcmV0dXJuIHMgLyAoTkcgLSAxKTsKfQo=",
 "js/kernel-worker.js": "Ly8gQSBoZWxwZXIgdGhyZWFkIGZvciBwb29sLmpzOiB3YWl0cyBmb3IgdGhlIG1haW4gdGhyZWFkIHRvIHN0YXJ0IGEgcGhhc2UsIHRha2VzIGNodW5rcyBvZiBpdCB1bnRpbCBub25lIGFyZQovLyBsZWZ0LCByZXBvcnRzIGRvbmUsIGFuZCB3YWl0cyBhZ2Fpbi4KaW1wb3J0IHsgcnVuQ2h1bmssIFBIQVNFUyB9IGZyb20gJy4va2VybmVscy5qcyc7Cgpjb25zdCBDX0dFTiA9IDAsIENfUEhBU0UgPSAxLCBDX05FWFQgPSAyLCBDX0RPTkUgPSAzLCBDX0NIVU5LUyA9IDQsIENfUkVBRFkgPSA1OwoKY29uc3QgV1QgPSB0eXBlb2YgcHJvY2VzcyAhPT0gJ3VuZGVmaW5lZCcgJiYgcHJvY2Vzcy52ZXJzaW9ucyAmJiBwcm9jZXNzLnZlcnNpb25zLm5vZGUgPyBhd2FpdCBpbXBvcnQoJ25vZGU6d29ya2VyX3RocmVhZHMnKSA6IG51bGw7CgpmdW5jdGlvbiBzZXJ2ZSh7IFYsIGN0bCB9KSB7CiAgbGV0IGdlbiA9IEF0b21pY3MubG9hZChjdGwsIENfR0VOKTsKICBBdG9taWNzLmFkZChjdGwsIENfUkVBRFksIDEpOyAvLyAoYWZ0ZXIgcmVhZGluZyBnZW46IHRoZSBtYWluIHRocmVhZCBzdGFydHMgdXNpbmcgdXMgb25seSB3aGVuIGFsbCBhcmUgcmVhZHkpCiAgZm9yICg7OykgewogICAgQXRvbWljcy53YWl0KGN0bCwgQ19HRU4sIGdlbik7CiAgICBnZW4gPSBBdG9taWNzLmxvYWQoY3RsLCBDX0dFTik7CiAgICBjb25zdCBwaGFzZSA9IFBIQVNFU1tBdG9taWNzLmxvYWQoY3RsLCBDX1BIQVNFKV0sIGNodW5rcyA9IEF0b21pY3MubG9hZChjdGwsIENfQ0hVTktTKTsKICAgIGZvciAobGV0IGs7IChrID0gQXRvbWljcy5hZGQoY3RsLCBDX05FWFQsIDEpKSA8IGNodW5rczspIHJ1bkNodW5rKFYsIHBoYXNlLCBrKTsKICAgIEF0b21pY3MuYWRkKGN0bCwgQ19ET05FLCAxKTsKICB9Cn0KCmlmIChXVCkgc2VydmUoV1Qud29ya2VyRGF0YSk7IC8vIE5vZGU6IHRoZSBhcnJheXMgY29tZSB3aXRoIHRoZSB0aHJlYWQKZWxzZSBzZWxmLm9ubWVzc2FnZSA9IChlKSA9PiBzZXJ2ZShlLmRhdGEpOyAvLyBicm93c2VyOiBpbiB0aGUgZmlyc3QgbWVzc2FnZQo=",
 "js/kernels.js": "Ly8gVGhlIGhlYXZ5IHBlci10aWNrIHdvcmsgb2YgdGhlIGxpdmluZyB3b3JsZCwgYXMgcHVyZSBmdW5jdGlvbnMgb3ZlciBmbGF0IGFycmF5cyBzbyBpdCBjYW4gcnVuIG9uIHNldmVyYWwKLy8gdGhyZWFkcyBhdCBvbmNlLiBFYWNoIGtlcm5lbCBjb21wdXRlcyBvbmUgcGhhc2UgZm9yIGEgcmFuZ2Ugb2YgY2VsbHMgKG9yIHZpcnVzZXMpLCByZWFkaW5nIG9ubHkgdGhlCi8vIHNoYXJlZCBpbnB1dCBhcnJheXMgYW5kIHdyaXRpbmcgb25seSB0aGF0IHJhbmdlJ3Mgb3V0cHV0cywgc28gYSBjZWxsJ3MgcmVzdWx0IGlzIHRoZSBzYW1lIHdoaWNoZXZlcgovLyB0aHJlYWQgY29tcHV0ZXMgaXQgYW5kIGhvd2V2ZXIgdGhlIHdvcmsgaXMgc3BsaXQ6IG9uZSB0aHJlYWQgb3IgZWlnaHQgZ2l2ZSBpZGVudGljYWwgd29ybGRzLgovLwovLyBQaGFzZXMgKHNlZSBMaWZlLnN0ZXApOiAnc3RlZXInIChsaWdodCwgc2Vuc2luZywgaGVhZGluZyksICdjb250YWN0JyAoY29sbGlzaW9uIHB1c2hlcyBhbmQgdGhlIGNvbnRhY3RzCi8vIHRoYXQgbWF5IGxlYWQgdG8gZWF0aW5nLCBnZW5lIHRyYW5zZmVyIG9yIHN5bWJpb3NpcyksICdyZWxheCcgKGdlbnRsZXIgZXh0cmEgcHVzaCBwYXNzKSwgJ3ZpcnVzJyAod2hpY2gKLy8gY2VsbHMgZWFjaCB2aXJ1cyB0b3VjaGVzKS4gVGhlIG1haW4gdGhyZWFkIHBhY2tzIHRoZSBpbnB1dHMsIHJ1bnMgdGhlIHBoYXNlIChpdHNlbGYgb3Igd2l0aCBoZWxwZXJzKSwKLy8gdGhlbiBhcHBsaWVzIHRoZSByZXN1bHRzIGluIGEgZml4ZWQgb3JkZXIuCgovLyBDZWxsIGZsYWdzLgpleHBvcnQgY29uc3QgRl9ERUFEID0gMSwgRl9NSVRPID0gMiwgRl9JTkZFQ1RFRCA9IDQsIEZfTk9ESVIgPSA4LCBGX0hVTlRTID0gMTYsCiAgRl9FQVRTID0gMzIsIEZfU0hPU1QgPSA2NCwgRl9TUEFSVCA9IDEyODsgLy8gY2FuIGVhdDsgY291bGQgaG9zdCAvIGJlY29tZSBhIG1pdG9jaG9uZHJpb24KLy8gT3V0cHV0IGZsYWdzIG9mIHRoZSBzdGVlciBwaGFzZS4KZXhwb3J0IGNvbnN0IE9fU0VOU0VEID0gMSwgT19SRVRISU5LID0gMjsKCi8vIEluZGljZXMgaW50byB0aGUgcGFyYW1ldGVyIGFycmF5IFAuCmV4cG9ydCBjb25zdCBQX1QgPSAwLCBQX1NFRUQgPSAxLCBQX01BWFIgPSAyLCBQX0hDUyA9IDMsIFBfSEdXID0gNCwgUF9IR0ggPSA1LCBQX0ZDUyA9IDYsIFBfRkdXID0gNywgUF9GR0ggPSA4LAogIFBfTElHSFQgPSA5LCBQX1NVTlNDQUxFID0gMTAsIFBfSCA9IDExLCBQX1ZFTlRHTE9XID0gMTIsIFBfTlZFTlRTID0gMTMsIFBfTiA9IDE0LCBQX05WID0gMTUsIFBfRE9NQUlOID0gMTYsCiAgUF9DSFVOS1MgPSAxNywgUF9TRUdDQVAgPSAxOCwgUF9WU0VHQ0FQID0gMTksIFBfSEdUID0gMjAsIFBfU1lNQiA9IDIxLCBQX1ZFTlRTID0gMjQ7IC8vIHZlbnRzOiB4LCB5LCBzdHJlbmd0aCBmcm9tIFBfVkVOVFMKZXhwb3J0IGNvbnN0IE1BWF9WRU5UUyA9IDMyOwoKLy8gQWxsb2NhdGUgdGhlIGFycmF5cyBmb3IgY2FwIGNlbGxzLCB2Y2FwIHZpcnVzZXMsIG5jIGhhc2ggc3F1YXJlcyBhbmQgZm4gZmllbGQgc3F1YXJlcy4gV2l0aCBzaGFyZWQgPSB0cnVlCi8vIHRoZXkgbGl2ZSBpbiBTaGFyZWRBcnJheUJ1ZmZlcnMgdGhhdCBoZWxwZXIgdGhyZWFkcyBjYW4gcmVhZCBhbmQgd3JpdGUuCmV4cG9ydCBmdW5jdGlvbiBtYWtlVmlld3MoY2FwLCB2Y2FwLCBuYywgZm4sIHN1bkxlbiwgY2h1bmtzLCBzaGFyZWQpIHsKICBjb25zdCBCID0gc2hhcmVkID8gU2hhcmVkQXJyYXlCdWZmZXIgOiBBcnJheUJ1ZmZlcjsKICBjb25zdCBmNjQgPSAobikgPT4gbmV3IEZsb2F0NjRBcnJheShuZXcgQihuICogOCkpLCBmMzIgPSAobikgPT4gbmV3IEZsb2F0MzJBcnJheShuZXcgQihuICogNCkpOwogIGNvbnN0IGkzMiA9IChuKSA9PiBuZXcgSW50MzJBcnJheShuZXcgQihuICogNCkpLCB1OCA9IChuKSA9PiBuZXcgVWludDhBcnJheShuZXcgQihuKSk7CiAgY29uc3Qgc2VnQ2FwID0gTWF0aC5jZWlsKChjYXAgKiAyNCkgLyBjaHVua3MpICsgNTEyLCB2c2VnQ2FwID0gTWF0aC5jZWlsKCh2Y2FwICogNikgLyBjaHVua3MpICsgMjU2OwogIHJldHVybiB7CiAgICBjYXAsIHZjYXAsIG5jLCBmbiwgY2h1bmtzLCBzZWdDYXAsIHZzZWdDYXAsCiAgICBQOiBmNjQoUF9WRU5UUyArIE1BWF9WRU5UUyAqIDMpLAogICAgWDogZjY0KGNhcCksIFk6IGY2NChjYXApLCBSOiBmNjQoY2FwKSwgQVJFQTogZjY0KGNhcCksIEVGRlBSRUQ6IGY2NChjYXApLCBTRU5TRTogZjY0KGNhcCksIE1PVDogZjY0KGNhcCksCiAgICBXQ0hFTTogZjY0KGNhcCksIFdPUkc6IGY2NChjYXApLCBXT1hZOiBmNjQoY2FwKSwgV0xJR0hUOiBmNjQoY2FwKSwgV1BSRVk6IGY2NChjYXApLCBXRkxFRTogZjY0KGNhcCksCiAgICBIRUFEOiBmNjQoY2FwKSwgUFg6IGY2NChjYXApLCBQWTogZjY0KGNhcCksIEZYOiBmNjQoY2FwKSwgRlk6IGY2NChjYXApLAogICAgRElSWDogZjY0KGNhcCksIERJUlk6IGY2NChjYXApLCBIRUFET1VUOiBmNjQoY2FwKSwgTElHSFRPVVQ6IGY2NChjYXApLCBQVVNIWDogZjY0KGNhcCksIFBVU0hZOiBmNjQoY2FwKSwKICAgIElEOiBpMzIoY2FwKSwgT1JHOiBpMzIoY2FwKSwgRkxBR1M6IHU4KGNhcCksIE9VVEY6IHU4KGNhcCksCiAgICBIU1RBUlQ6IGkzMihuYyArIDEpLCBISVRFTVM6IGkzMihjYXApLCBQU1RBUlQ6IGkzMihuYyArIDEpLCBQSVRFTVM6IGkzMihjYXApLCBQSURYOiBpMzIoY2FwKSwKICAgIEZDSEVNOiBmMzIoZm4pLCBGT1JHOiBmMzIoZm4pLCBGT1hZOiBmMzIoZm4pLCBTVU46IGYzMihzdW5MZW4pLAogICAgVlg6IGY2NCh2Y2FwKSwgVlk6IGY2NCh2Y2FwKSwgVkZMQUdTOiB1OCh2Y2FwKSwgLy8gYml0IDE6IHNlYXJjaGVzIHRoaXMgdGljaywgYml0IDI6IGluZmVjdHMgZXVrYXJ5b3RlcwogICAgRVY6IGkzMihzZWdDYXAgKiBjaHVua3MpLCBFVk46IGkzMihjaHVua3MpLCBWQzogaTMyKHZzZWdDYXAgKiBjaHVua3MpLCBWQ046IGkzMihjaHVua3MpLCBWU1RBUlQ6IGkzMih2Y2FwKSwgVkNPVU5UOiBpMzIodmNhcCksCiAgfTsKfQoKLy8gQSBoYXNoIG9mIChzZWVkLCB0aWNrLCBpZCkgdG8gWzAsIDEpOiBlYWNoIGNlbGwncyBvd24gcmFuZG9tIHN0cmVhbSwgaW5kZXBlbmRlbnQgb2YgdGhlIG9yZGVyIGNlbGxzIGFyZQovLyBwcm9jZXNzZWQgaW4gKHRoZSBzaGFyZWQgcmFuZG9tIGdlbmVyYXRvciB3b3VsZCBkZXBlbmQgb24gaXQpLgpleHBvcnQgZnVuY3Rpb24gaHJhbmQoc2VlZCwgdCwgaWQpIHsKICBsZXQgaCA9IE1hdGguaW11bChzZWVkIF4gMHg5ZTM3NzliOSwgMHg4NWViY2E2YikgXiBNYXRoLmltdWwodCArIDB4NjMyYmU1YWIsIDB4YzJiMmFlMzUpIF4gTWF0aC5pbXVsKGlkICsgMSwgMHgyN2Q0ZWIyZik7CiAgaCBePSBoID4+PiAxNTsgaCA9IE1hdGguaW11bChoLCAweDJjMWIzYzZkKTsgaCBePSBoID4+PiAxMjsgaCA9IE1hdGguaW11bChoLCAweDI5N2EyZDM5KTsgaCBePSBoID4+PiAxNTsKICByZXR1cm4gKGggPj4+IDApIC8gNDI5NDk2NzI5NjsKfQoKLy8gVGhlIGdyaWQgc3F1YXJlcyBhIHF1ZXJ5IGJveCBjb3ZlcnMgKHNhbWUgcnVsZSBhcyBTcGF0aWFsSGFzaC5ib3gpLCB3cml0dGVuIGludG8gb3V0WzAuLjNdLgpmdW5jdGlvbiBib3goUCwgeCwgeSwgciwgb3V0KSB7CiAgY29uc3QgY3MgPSBQW1BfSENTXSwgZ3cgPSBQW1BfSEdXXSwgZ2ggPSBQW1BfSEdIXTsKICBsZXQgeDAgPSBNYXRoLmZsb29yKCh4IC0gcikgLyBjcyksIHgxID0gTWF0aC5mbG9vcigoeCArIHIpIC8gY3MpOwogIGxldCB5MCA9IE1hdGguZmxvb3IoKHkgLSByKSAvIGNzKSwgeTEgPSBNYXRoLmZsb29yKCh5ICsgcikgLyBjcyk7CiAgaWYgKHgwIDwgMCkgeDAgPSAwOyBpZiAoeTAgPCAwKSB5MCA9IDA7CiAgaWYgKHgxID49IGd3KSB4MSA9IGd3IC0gMTsgaWYgKHkxID49IGdoKSB5MSA9IGdoIC0gMTsKICBvdXRbMF0gPSB4MDsgb3V0WzFdID0geDE7IG91dFsyXSA9IHkwOyBvdXRbM10gPSB5MTsKfQoKLy8gR3JhZGllbnQgb2YgYSBmaWVsZCAoc2FtZSBydWxlIGFzIEZpZWxkcy5ncmFkKSwgd3JpdHRlbiBpbnRvIGdbMC4uMV0uCmZ1bmN0aW9uIGdyYWQoViwgZiwgeCwgeSwgZykgewogIGNvbnN0IFAgPSBWLlAsIGNzID0gUFtQX0ZDU10sIGd3ID0gUFtQX0ZHV10sIGdoID0gUFtQX0ZHSF07CiAgbGV0IGN4ID0gKHggLyBjcykgfCAwLCBjeSA9ICh5IC8gY3MpIHwgMDsKICBpZiAoY3ggPCAxKSBjeCA9IDE7IGVsc2UgaWYgKGN4ID4gZ3cgLSAyKSBjeCA9IGd3IC0gMjsKICBpZiAoY3kgPCAxKSBjeSA9IDE7IGVsc2UgaWYgKGN5ID4gZ2ggLSAyKSBjeSA9IGdoIC0gMjsKICBjb25zdCBpID0gY3kgKiBndyArIGN4OwogIGNvbnN0IGd4ID0gZltpICsgMV0gLSBmW2kgLSAxXSwgZ3kgPSBmW2kgKyBnd10gLSBmW2kgLSBnd107CiAgY29uc3QgbSA9IE1hdGguaHlwb3QoZ3gsIGd5KTsKICBpZiAobSA8IDFlLTYpIHsgZ1swXSA9IDA7IGdbMV0gPSAwOyByZXR1cm47IH0KICBjb25zdCByZWwgPSBNYXRoLm1pbigxLCAobSAvIChmW2ldICsgMC41KSkgKiA1KTsKICBnWzBdID0gKGd4IC8gbSkgKiByZWw7IGdbMV0gPSAoZ3kgLyBtKSAqIHJlbDsKfQoKLy8gTGlnaHQgYXQgYSBwb2ludCAoc2FtZSBydWxlIGFzIFdvcmxkLmxpZ2h0QXQpLgpmdW5jdGlvbiBsaWdodEF0KFAsIFNVTiwgeCwgeSkgewogIGxldCBpID0gKHkgKiBQW1BfU1VOU0NBTEVdKSB8IDA7CiAgaWYgKGkgPCAwKSBpID0gMDsgZWxzZSBpZiAoaSA+PSBTVU4ubGVuZ3RoKSBpID0gU1VOLmxlbmd0aCAtIDE7CiAgbGV0IGwgPSBQW1BfTElHSFRdICogU1VOW2ldOwogIGlmICh5ID4gUFtQX0hdIC0gMTQwKSB7CiAgICBmb3IgKGxldCBrID0gMCwgbnYgPSBQW1BfTlZFTlRTXTsgayA8IG52OyBrKyspIHsKICAgICAgY29uc3QgdnggPSBQW1BfVkVOVFMgKyBrICogM10sIHZ5ID0gUFtQX1ZFTlRTICsgayAqIDMgKyAxXSwgdnMgPSBQW1BfVkVOVFMgKyBrICogMyArIDJdOwogICAgICBjb25zdCBkeCA9IHggLSB2eCwgZHkgPSB5IC0gdnksIGQyID0gZHggKiBkeCArIGR5ICogZHk7CiAgICAgIGlmIChkMiA8IDIyNTAwKSBsICs9IFBbUF9WRU5UR0xPV10gKiB2cyAqICgxIC0gTWF0aC5zcXJ0KGQyKSAvIDE1MCk7CiAgICB9CiAgfQogIHJldHVybiBsOwp9CgovLyAtLS0gc3RlZXI6IGxpZ2h0IGF0IGVhY2ggY2VsbCwgYW5kIGZvciBtb3RpbGUgY2VsbHMgdGhlaXIgc2Vuc2luZyBhbmQgKGV2ZXJ5IHRoaXJkIHRpY2spIGEgbmV3IGhlYWRpbmcuCmV4cG9ydCBmdW5jdGlvbiBzdGVlcktlcm5lbChWLCBpMCwgaTEpIHsKICBjb25zdCBQID0gVi5QLCB0ID0gUFtQX1RdLCBzZWVkID0gUFtQX1NFRURdLCBuID0gUFtQX05dOwogIGNvbnN0IHsgWCwgWSwgUiwgT1JHLCBGTEFHUywgRUZGUFJFRCwgU0VOU0UsIE1PVCwgSUQsIEhFQUQsIFBYLCBQWSwgRlgsIEZZLCBPVVRGIH0gPSBWOwogIGNvbnN0IGJ4ID0gWzAsIDAsIDAsIDBdLCBnID0gWzAsIDBdOwogIGZvciAobGV0IGkgPSBpMDsgaSA8IGkxOyBpKyspIHsKICAgIE9VVEZbaV0gPSAwOwogICAgaWYgKEZMQUdTW2ldICYgRl9ERUFEKSBjb250aW51ZTsKICAgIGNvbnN0IHggPSBYW2ldLCB5ID0gWVtpXTsKICAgIFYuTElHSFRPVVRbaV0gPSBsaWdodEF0KFAsIFYuU1VOLCB4LCB5KTsKICAgIGlmIChNT1RbaV0gPCAwLjAzKSBjb250aW51ZTsKICAgIGNvbnN0IGlkID0gSURbaV07CiAgICAvLyBTZW5zaW5nIHByZXkgYW5kIHByZWRhdG9ycyAob25seSBsaW5lYWdlcyB0aGF0IHJlYWN0IHRvIHRoZW0sIGV2ZXJ5IHNpeHRoIHRpY2spLgogICAgaWYgKChpZCArIHQpICUgNiA9PT0gMCAmJiAoTWF0aC5hYnMoVi5XUFJFWVtpXSkgPiAwLjEgfHwgTWF0aC5hYnMoVi5XRkxFRVtpXSkgPiAwLjEpKSB7CiAgICAgIGxldCBweCA9IDAsIHB5ID0gMCwgZnggPSAwLCBmeSA9IDAsIGJwID0gMWU5LCBiZiA9IDFlOTsKICAgICAgY29uc3QgcyA9IFNFTlNFW2ldLCBzMiA9IHMgKiBzLCByID0gUltpXSwgb3JnID0gT1JHW2ldLCBodW50cyA9IChGTEFHU1tpXSAmIEZfSFVOVFMpICE9PSAwOwogICAgICAvLyBOb24taHVudGVycyBvbmx5IHJlYWN0IHRvIHByZWRhdG9yczogd2FsayB0aGUgcHJlZGF0b3IgZ3JpZCAoc2FtZSBzcXVhcmVzLCBzYW1lIG9yZGVyKS4KICAgICAgY29uc3Qgc3RhcnQgPSBodW50cyA/IFYuSFNUQVJUIDogVi5QU1RBUlQsIGl0ZW1zID0gaHVudHMgPyBWLkhJVEVNUyA6IFYuUElURU1TLCBndyA9IFBbUF9IR1ddOwogICAgICBib3goUCwgeCwgeSwgcywgYngpOwogICAgICBmb3IgKGxldCBjeSA9IGJ4WzJdOyBjeSA8PSBieFszXTsgY3krKykgewogICAgICAgIGZvciAobGV0IHEgPSBjeSAqIGd3ICsgYnhbMF0sIHFlID0gY3kgKiBndyArIGJ4WzFdOyBxIDw9IHFlOyBxKyspIHsKICAgICAgICAgIGZvciAobGV0IGsgPSBzdGFydFtxXSwgZSA9IHN0YXJ0W3EgKyAxXTsgayA8IGU7IGsrKykgewogICAgICAgICAgICBjb25zdCBqID0gaHVudHMgPyBpdGVtc1trXSA6IFYuUElEWFtpdGVtc1trXV07CiAgICAgICAgICAgIGlmIChqID09PSBpIHx8IChGTEFHU1tqXSAmIEZfREVBRCkgfHwgT1JHW2pdID09PSBvcmcpIGNvbnRpbnVlOwogICAgICAgICAgICBjb25zdCBkeCA9IFhbal0gLSB4LCBkeSA9IFlbal0gLSB5LCBkMiA9IGR4ICogZHggKyBkeSAqIGR5OwogICAgICAgICAgICBpZiAoZDIgPiBzMikgY29udGludWU7CiAgICAgICAgICAgIGlmIChodW50cyAmJiBSW2pdICogMS4yIDwgciAmJiBkMiA8IGJwKSB7IGJwID0gZDI7IHB4ID0gZHg7IHB5ID0gZHk7IH0KICAgICAgICAgICAgaWYgKEVGRlBSRURbal0gPiAwLjA1ICYmIFJbal0gPiByICogMS4yICYmIGQyIDwgYmYpIHsgYmYgPSBkMjsgZnggPSAtZHg7IGZ5ID0gLWR5OyB9CiAgICAgICAgICB9CiAgICAgICAgfQogICAgICB9CiAgICAgIGxldCBtID0gTWF0aC5oeXBvdChweCwgcHkpOyBpZiAobSA+IDApIHsgcHggLz0gbTsgcHkgLz0gbTsgfQogICAgICBtID0gTWF0aC5oeXBvdChmeCwgZnkpOyBpZiAobSA+IDApIHsgZnggLz0gbTsgZnkgLz0gbTsgfQogICAgICBQWFtpXSA9IHB4OyBQWVtpXSA9IHB5OyBGWFtpXSA9IGZ4OyBGWVtpXSA9IGZ5OwogICAgICBPVVRGW2ldIHw9IDE7CiAgICB9CiAgICAvLyBHcmFkaWVudHMgY2hhbmdlIHNsb3dseTogcmV0aGluayB0aGUgaGVhZGluZyBldmVyeSB0aGlyZCB0aWNrIChhbmQgdGhlIGZpcnN0IHRpbWUpLgogICAgaWYgKChpZCArIHQpICUgMyA9PT0gMCB8fCAoRkxBR1NbaV0gJiBGX05PRElSKSkgewogICAgICBsZXQgZHggPSAwLCBkeSA9IDA7CiAgICAgIGdyYWQoViwgVi5GQ0hFTSwgeCwgeSwgZyk7IGR4ICs9IGdbMF0gKiBWLldDSEVNW2ldOyBkeSArPSBnWzFdICogVi5XQ0hFTVtpXTsKICAgICAgZ3JhZChWLCBWLkZPUkcsIHgsIHksIGcpOyBkeCArPSBnWzBdICogVi5XT1JHW2ldOyBkeSArPSBnWzFdICogVi5XT1JHW2ldOwogICAgICBncmFkKFYsIFYuRk9YWSwgeCwgeSwgZyk7IGR4ICs9IGdbMF0gKiBWLldPWFlbaV07IGR5ICs9IGdbMV0gKiBWLldPWFlbaV07CiAgICAgIGR5IC09IFYuV0xJR0hUW2ldICogMC44OyAvLyBwaG90b3RheGlzOiBsaWdodCBjb21lcyBmcm9tIGFib3ZlCiAgICAgIGR4ICs9IFBYW2ldICogVi5XUFJFWVtpXSArIEZYW2ldICogVi5XRkxFRVtpXTsKICAgICAgZHkgKz0gUFlbaV0gKiBWLldQUkVZW2ldICsgRllbaV0gKiBWLldGTEVFW2ldOwogICAgICBjb25zdCBoZWFkaW5nID0gSEVBRFtpXSArIChocmFuZChzZWVkLCB0LCBpZCkgLSAwLjUpICogMC41OwogICAgICBkeCArPSBNYXRoLmNvcyhoZWFkaW5nKSAqIDAuMzsKICAgICAgZHkgKz0gTWF0aC5zaW4oaGVhZGluZykgKiAwLjM7CiAgICAgIGNvbnN0IG0gPSBNYXRoLmh5cG90KGR4LCBkeSkgfHwgMTsKICAgICAgVi5IRUFET1VUW2ldID0gaGVhZGluZzsgVi5ESVJYW2ldID0gZHggLyBtOyBWLkRJUllbaV0gPSBkeSAvIG07CiAgICAgIE9VVEZbaV0gfD0gMjsKICAgIH0KICB9Cn0KCi8vIC0tLSBjb250YWN0IC8gcmVsYXg6IGVhY2ggY2VsbCdzIHB1c2ggZnJvbSBldmVyeSBuZWlnaGJvdXIgaXQgb3ZlcmxhcHMgKGFsbCBjb21wdXRlZCBmcm9tIHRoZSBzYW1lCi8vIHBvc2l0aW9ucywgdGhlbiBhcHBsaWVkIHRvZ2V0aGVyKS4gVGhlIGNvbnRhY3QgcGFzcyBhbHNvIGxpc3RzIHRoZSB0b3VjaGluZyBwYWlycyBmcm9tIGRpZmZlcmVudCBvcmdhbmlzbXMKLy8gd2hlcmUgc29tZXRoaW5nIGNhbiBoYXBwZW46IG9uZSBjb3VsZCBlYXQgdGhlIG90aGVyLCB0aGV5IGNvdWxkIG1lcmdlIChzeW1iaW9zaXMpLCBvciBnZW5lcyBwYXNzIGJldHdlZW4KLy8gdGhlbSAoZGVjaWRlZCBoZXJlLCBmcm9tIHRoZSBwYWlyJ3Mgb3duIHJhbmRvbSBzdHJlYW0pLiBFYWNoIGVudHJ5IGlzIChpLCBqLCBnZW5lIHRyYW5zZmVyPyk7IHRoZXkgYXJlCi8vIHJlc29sdmVkIG9uZSBieSBvbmUgYWZ0ZXJ3YXJkcywgaW4gdGhpcyBvcmRlci4KZXhwb3J0IGZ1bmN0aW9uIGNvbnRhY3RLZXJuZWwoViwgaTAsIGkxLCByZWxheCwgY2h1bmspIHsKICBjb25zdCBQID0gVi5QLCBtYXhSID0gUFtQX01BWFJdLCBndyA9IFBbUF9IR1ddLCB7IFgsIFksIFIsIEFSRUEsIE9SRywgRkxBR1MsIElELCBIU1RBUlQsIEhJVEVNUywgUFVTSFgsIFBVU0hZLCBFViB9ID0gVjsKICBjb25zdCBzZWVkID0gUFtQX1NFRURdLCB0ID0gUFtQX1RdLCBoZ3RQID0gUFtQX0hHVF0sIHN5bWIgPSBQW1BfU1lNQl0gIT09IDA7CiAgY29uc3QgYnggPSBbMCwgMCwgMCwgMF0sIHNlZ0NhcCA9IFBbUF9TRUdDQVBdLCBiYXNlID0gY2h1bmsgKiBzZWdDYXA7CiAgbGV0IG5lID0gMCwgb3ZlcmZsb3cgPSBmYWxzZTsKICBmb3IgKGxldCBpID0gaTA7IGkgPCBpMTsgaSsrKSB7CiAgICBsZXQgcHVzaHggPSAwLCBwdXNoeSA9IDA7CiAgICBpZiAoRkxBR1NbaV0gJiBGX0RFQUQpIHsgUFVTSFhbaV0gPSAwOyBQVVNIWVtpXSA9IDA7IGNvbnRpbnVlOyB9CiAgICBjb25zdCBheCA9IFhbaV0sIGF5ID0gWVtpXSwgYXIgPSBSW2ldLCBhYXJlYSA9IEFSRUFbaV0sIG9yZyA9IE9SR1tpXTsKICAgIGJveChQLCBheCwgYXksIGFyICsgbWF4UiwgYngpOwogICAgZm9yIChsZXQgY3kgPSBieFsyXTsgY3kgPD0gYnhbM107IGN5KyspIHsKICAgICAgZm9yIChsZXQgcSA9IGN5ICogZ3cgKyBieFswXSwgcWUgPSBjeSAqIGd3ICsgYnhbMV07IHEgPD0gcWU7IHErKykgewogICAgICAgIGZvciAobGV0IGsgPSBIU1RBUlRbcV0sIGUgPSBIU1RBUlRbcSArIDFdOyBrIDwgZTsgaysrKSB7CiAgICAgICAgICBjb25zdCBqID0gSElURU1TW2tdOwogICAgICAgICAgaWYgKGogPT09IGkgfHwgKEZMQUdTW2pdICYgRl9ERUFEKSkgY29udGludWU7CiAgICAgICAgICBjb25zdCBkeCA9IFhbal0gLSBheCwgZHkgPSBZW2pdIC0gYXksIGQyID0gZHggKiBkeCArIGR5ICogZHk7CiAgICAgICAgICBjb25zdCByciA9IHJlbGF4ID8gKGFyICsgUltqXSkgKiAwLjk3IDogYXIgKyBSW2pdOwogICAgICAgICAgaWYgKGQyID49IHJyICogcnIpIGNvbnRpbnVlOwogICAgICAgICAgY29uc3QgZCA9IE1hdGguc3FydChkMikgfHwgMC4wMSwgcyA9IGFhcmVhICsgQVJFQVtqXTsKICAgICAgICAgIGNvbnN0IG92ID0gKHJyIC0gZCkgKiAocmVsYXggPyAwLjcgOiAwLjgpOyAvLyBtZW1icmFuZXMgYmFyZWx5IGludGVycGVuZXRyYXRlCiAgICAgICAgICBwdXNoeCAtPSAoZHggLyBkKSAqIG92ICogKEFSRUFbal0gLyBzKTsgcHVzaHkgLT0gKGR5IC8gZCkgKiBvdiAqIChBUkVBW2pdIC8gcyk7CiAgICAgICAgICBpZiAoIXJlbGF4ICYmIGogPiBpICYmIE9SR1tqXSAhPT0gb3JnKSB7CiAgICAgICAgICAgIGNvbnN0IGZpID0gRkxBR1NbaV0sIGZqID0gRkxBR1Nbal07CiAgICAgICAgICAgIGNvbnN0IGhndCA9IGhyYW5kKHNlZWQsIHQsIE1hdGguaW11bChJRFtpXSwgNzM4NTYwOTMpIF4gTWF0aC5pbXVsKElEW2pdLCAxOTM0OTY2MykpIDwgaGd0UCA/IDEgOiAwOwogICAgICAgICAgICBjb25zdCBlYXRzID0gKChmaSB8IGZqKSAmIEZfRUFUUykgIT09IDA7CiAgICAgICAgICAgIGNvbnN0IG1lcmdlID0gc3ltYiAmJiAoKChmaSAmIEZfU0hPU1QpICYmIChmaiAmIEZfU1BBUlQpKSB8fCAoKGZqICYgRl9TSE9TVCkgJiYgKGZpICYgRl9TUEFSVCkpKTsKICAgICAgICAgICAgaWYgKGhndCB8fCBlYXRzIHx8IG1lcmdlKSB7CiAgICAgICAgICAgICAgaWYgKG5lICsgMyA+IHNlZ0NhcCkgb3ZlcmZsb3cgPSB0cnVlOwogICAgICAgICAgICAgIGVsc2UgeyBFVltiYXNlICsgbmVdID0gaTsgRVZbYmFzZSArIG5lICsgMV0gPSBqOyBFVltiYXNlICsgbmUgKyAyXSA9IGhndDsgbmUgKz0gMzsgfQogICAgICAgICAgICB9CiAgICAgICAgICB9CiAgICAgICAgfQogICAgICB9CiAgICB9CiAgICBQVVNIWFtpXSA9IHB1c2h4OyBQVVNIWVtpXSA9IHB1c2h5OwogIH0KICBpZiAoIXJlbGF4KSBWLkVWTltjaHVua10gPSBvdmVyZmxvdyA/IC0xIDogbmU7Cn0KCi8vIC0tLSB2aXJ1czogZm9yIGVhY2ggdmlydXMgc2VhcmNoaW5nIHRoaXMgdGljaywgdGhlIGNlbGxzIGl0IHRvdWNoZXMgKGluIGdyaWQgb3JkZXIpLCBza2lwcGluZyBjZWxscwovLyBhbHJlYWR5IGRlYWQgb3IgaW5mZWN0ZWQgYW5kLCB3aXRoIGRvbWFpbnMgb24sIGNlbGxzIG9mIHRoZSBvdGhlciBkb21haW4uIEluZmVjdGlvbnMgYXJlIHJlc29sdmVkCi8vIGFmdGVyd2FyZHMsIG9uZSB2aXJ1cyBhdCBhIHRpbWUuCmV4cG9ydCBmdW5jdGlvbiB2aXJ1c0tlcm5lbChWLCB2MCwgdjEsIGNodW5rKSB7CiAgY29uc3QgUCA9IFYuUCwgbWF4UiA9IFBbUF9NQVhSXSwgZ3cgPSBQW1BfSEdXXSwgZG9tYWluID0gUFtQX0RPTUFJTl0gIT09IDA7CiAgY29uc3QgeyBYLCBZLCBSLCBGTEFHUywgSFNUQVJULCBISVRFTVMsIFZYLCBWWSwgVkZMQUdTLCBWQywgVlNUQVJULCBWQ09VTlQgfSA9IFY7CiAgY29uc3QgYnggPSBbMCwgMCwgMCwgMF0sIGNhcCA9IFBbUF9WU0VHQ0FQXSwgYmFzZSA9IGNodW5rICogY2FwOwogIGxldCBuYyA9IDAsIG92ZXJmbG93ID0gZmFsc2U7CiAgZm9yIChsZXQgazAgPSB2MDsgazAgPCB2MTsgazArKykgewogICAgVlNUQVJUW2swXSA9IGJhc2UgKyBuYzsgVkNPVU5UW2swXSA9IDA7CiAgICBpZiAoIShWRkxBR1NbazBdICYgMSkpIGNvbnRpbnVlOwogICAgY29uc3QgdnggPSBWWFtrMF0sIHZ5ID0gVllbazBdLCB2ZXVrID0gKFZGTEFHU1trMF0gJiAyKSAhPT0gMDsKICAgIGJveChQLCB2eCwgdnksIG1heFIgKyAxLCBieCk7CiAgICBmb3IgKGxldCBjeSA9IGJ4WzJdOyBjeSA8PSBieFszXTsgY3krKykgewogICAgICBmb3IgKGxldCBxID0gY3kgKiBndyArIGJ4WzBdLCBxZSA9IGN5ICogZ3cgKyBieFsxXTsgcSA8PSBxZTsgcSsrKSB7CiAgICAgICAgZm9yIChsZXQgayA9IEhTVEFSVFtxXSwgZSA9IEhTVEFSVFtxICsgMV07IGsgPCBlOyBrKyspIHsKICAgICAgICAgIGNvbnN0IGogPSBISVRFTVNba10sIGYgPSBGTEFHU1tqXTsKICAgICAgICAgIGlmIChmICYgKEZfREVBRCB8IEZfSU5GRUNURUQpKSBjb250aW51ZTsKICAgICAgICAgIGlmIChkb21haW4gJiYgdmV1ayAhPT0gKChmICYgRl9NSVRPKSAhPT0gMCkpIGNvbnRpbnVlOwogICAgICAgICAgY29uc3QgZHggPSBYW2pdIC0gdngsIGR5ID0gWVtqXSAtIHZ5OwogICAgICAgICAgaWYgKGR4ICogZHggKyBkeSAqIGR5ID4gKFJbal0gKyAwLjEpICogKFJbal0gKyAwLjEpKSBjb250aW51ZTsKICAgICAgICAgIGlmIChuYyArIDEgPiBjYXApIHsgb3ZlcmZsb3cgPSB0cnVlOyBjb250aW51ZTsgfQogICAgICAgICAgVkNbYmFzZSArIG5jKytdID0gajsgVkNPVU5UW2swXSsrOwogICAgICAgIH0KICAgICAgfQogICAgfQogIH0KICBWLlZDTltjaHVua10gPSBvdmVyZmxvdyA/IC0xIDogbmM7Cn0KCmV4cG9ydCBmdW5jdGlvbiBydW5DaHVuayhWLCBwaGFzZSwgY2h1bmspIHsKICBjb25zdCBQID0gVi5QLCBLID0gUFtQX0NIVU5LU10sIHRvdGFsID0gcGhhc2UgPT09ICd2aXJ1cycgPyBQW1BfTlZdIDogUFtQX05dOwogIGNvbnN0IGkwID0gTWF0aC5mbG9vcigoY2h1bmsgKiB0b3RhbCkgLyBLKSwgaTEgPSBNYXRoLmZsb29yKCgoY2h1bmsgKyAxKSAqIHRvdGFsKSAvIEspOwogIGlmIChwaGFzZSA9PT0gJ3N0ZWVyJykgc3RlZXJLZXJuZWwoViwgaTAsIGkxKTsKICBlbHNlIGlmIChwaGFzZSA9PT0gJ2NvbnRhY3QnKSBjb250YWN0S2VybmVsKFYsIGkwLCBpMSwgZmFsc2UsIGNodW5rKTsKICBlbHNlIGlmIChwaGFzZSA9PT0gJ3JlbGF4JykgY29udGFjdEtlcm5lbChWLCBpMCwgaTEsIHRydWUsIGNodW5rKTsKICBlbHNlIGlmIChwaGFzZSA9PT0gJ3ZpcnVzJykgdmlydXNLZXJuZWwoViwgaTAsIGkxLCBjaHVuayk7Cn0KZXhwb3J0IGNvbnN0IFBIQVNFUyA9IFsnc3RlZXInLCAnY29udGFjdCcsICdyZWxheCcsICd2aXJ1cyddOwo=",
 "js/life.js": "...",
 "js/pool.js": "Ly8gQSBzbWFsbCBwb29sIG9mIGhlbHBlciB0aHJlYWRzIGZvciB0aGUga2VybmVscyBpbiBrZXJuZWxzLmpzLiBUaGUgbWFpbiB0aHJlYWQgcGFja3MgYSBwaGFzZSdzIGlucHV0cyBpbnRvCi8vIHNoYXJlZCBhcnJheXMsIHRoZW4gaXQgYW5kIHRoZSBoZWxwZXJzIHRha2UgY2h1bmtzIG9mIHRoZSB3b3JrIGZyb20gYSBzaGFyZWQgY291bnRlciB1bnRpbCBhbGwgYXJlIGRvbmUuCi8vIFJlc3VsdHMgZG9uJ3QgZGVwZW5kIG9uIGhvdyBtYW55IHRocmVhZHMgdGhlcmUgYXJlIChlYWNoIGNlbGwncyByZXN1bHQgaXMgY29tcHV0ZWQgdGhlIHNhbWUgd2F5IGFuZCB0aGUKLy8gbWFpbiB0aHJlYWQgbWVyZ2VzIGNodW5rcyBpbiBvcmRlciksIHNvIHdpdGhvdXQgc2hhcmVkIG1lbW9yeSBldmVyeXRoaW5nIHNpbXBseSBydW5zIG9uIHRoZSBtYWluIHRocmVhZC4KaW1wb3J0IHsgbWFrZVZpZXdzLCBydW5DaHVuaywgUEhBU0VTLCBQX0NIVU5LUyB9IGZyb20gJy4va2VybmVscy5qcyc7CmltcG9ydCB7IENGRyB9IGZyb20gJy4vY29uZmlnLmpzJzsKCi8vIE5vZGUgaGFzIGl0cyBvd24gdGhyZWFkcyAod29ya2VyX3RocmVhZHMpOyBicm93c2VycyBoYXZlIFdlYiBXb3JrZXJzLgpjb25zdCBXVCA9IHR5cGVvZiBwcm9jZXNzICE9PSAndW5kZWZpbmVkJyAmJiBwcm9jZXNzLnZlcnNpb25zICYmIHByb2Nlc3MudmVyc2lvbnMubm9kZSA/IGF3YWl0IGltcG9ydCgnbm9kZTp3b3JrZXJfdGhyZWFkcycpIDogbnVsbDsKCmNvbnN0IENfR0VOID0gMCwgQ19QSEFTRSA9IDEsIENfTkVYVCA9IDIsIENfRE9ORSA9IDMsIENfQ0hVTktTID0gNCwgQ19SRUFEWSA9IDU7CmNvbnN0IENIVU5LUyA9IDQ4OyAvLyBtb3JlIGNodW5rcyB0aGFuIHRocmVhZHMsIHNvIGRlbnNlIHBhdGNoZXMgZG9uJ3QgbGVhdmUgdGhyZWFkcyBpZGxlCgpsZXQgcG9vbDsgLy8gY3JlYXRlZCBvbiBmaXJzdCB1c2UKCi8vIEhlbHBlciB0aHJlYWRzIHRvIHVzZTogQ0ZHLlRIUkVBRFMsIG9yICgwID0gYXV0bykgYWxsIGJ1dCBvbmUgY29yZSBvbiBhIGJyb3dzZXIgcGFnZTsgbm9uZSBpbiBOb2RlIG9yIGluc2lkZQovLyBhIHdvcmtlciAodGhlIHRvdXIncyBoaXN0b3J5IHNlYXJjaCwgYmF0Y2ggcnVucyB0aGF0IGFscmVhZHkgcnVuIG9uZSB3b3JsZCBwZXIgY29yZSkuCmZ1bmN0aW9uIGhlbHBlckNvdW50KCkgewogIGlmICh0eXBlb2YgU2hhcmVkQXJyYXlCdWZmZXIgPT09ICd1bmRlZmluZWQnIHx8IHR5cGVvZiBBdG9taWNzID09PSAndW5kZWZpbmVkJykgcmV0dXJuIDA7CiAgaWYgKHR5cGVvZiB3aW5kb3cgPT09ICd1bmRlZmluZWQnIHx8IHR5cGVvZiBkb2N1bWVudCA9PT0gJ3VuZGVmaW5lZCcpIHJldHVybiBDRkcuVEhSRUFEUyA+IDEgPyBDRkcuVEhSRUFEUyAtIDEgOiAwOwogIGlmICghZ2xvYmFsVGhpcy5jcm9zc09yaWdpbklzb2xhdGVkKSByZXR1cm4gMDsgLy8gc2hhcmVkIG1lbW9yeSBuZWVkcyB0aGUgcGFnZSB0byBiZSBjcm9zcy1vcmlnaW4gaXNvbGF0ZWQKICBpZiAoQ0ZHLlRIUkVBRFMgPiAwKSByZXR1cm4gQ0ZHLlRIUkVBRFMgLSAxOwogIHJldHVybiBNYXRoLm1heCgwLCBNYXRoLm1pbig3LCAobmF2aWdhdG9yLmhhcmR3YXJlQ29uY3VycmVuY3kgfHwgMikgLSAxKSk7Cn0KCmZ1bmN0aW9uIGNyZWF0ZVBvb2woY2FwLCB2Y2FwLCBuYywgZm4sIHN1bkxlbikgewogIGNvbnN0IGhlbHBlcnMgPSBoZWxwZXJDb3VudCgpOwogIGNvbnN0IHNoYXJlZCA9IGhlbHBlcnMgPiAwOwogIGNvbnN0IFYgPSBtYWtlVmlld3MoY2FwLCB2Y2FwLCBuYywgZm4sIHN1bkxlbiwgQ0hVTktTLCBzaGFyZWQpOwogIGNvbnN0IHAgPSB7IFYsIGhlbHBlcnMsIHNoYXJlZCwgY2FwLCB2Y2FwLCBuYywgZm4sIHN1bkxlbiwgd29ya2VyczogW10sIGN0bDogbnVsbCB9OwogIGlmICghc2hhcmVkKSByZXR1cm4gcDsKICBwLmN0bCA9IG5ldyBJbnQzMkFycmF5KG5ldyBTaGFyZWRBcnJheUJ1ZmZlcig2NCkpOwogIGZvciAobGV0IGsgPSAwOyBrIDwgaGVscGVyczsgaysrKSB7CiAgICBjb25zdCB1cmwgPSBuZXcgVVJMKCcuL2tlcm5lbC13b3JrZXIuanMnLCBpbXBvcnQubWV0YS51cmwpOwogICAgY29uc3QgdyA9IFdUID8gbmV3IFdULldvcmtlcih1cmwsIHsgd29ya2VyRGF0YTogeyBWLCBjdGw6IHAuY3RsIH0gfSkgOiBuZXcgV29ya2VyKHVybCwgeyB0eXBlOiAnbW9kdWxlJyB9KTsKICAgIGlmIChXVCkgdy51bnJlZigpOyBlbHNlIHcucG9zdE1lc3NhZ2UoeyBWLCBjdGw6IHAuY3RsIH0pOwogICAgcC53b3JrZXJzLnB1c2godyk7CiAgfQogIHJldHVybiBwOwp9CgovLyBUaGUgcG9vbCdzIGFycmF5cywgcmVjcmVhdGVkIGJpZ2dlciAod2l0aCBmcmVzaCBoZWxwZXJzKSBpZiB0aGlzIHdvcmxkIGRvZXNuJ3QgZml0LgpleHBvcnQgZnVuY3Rpb24gdmlld3MoY2FwLCB2Y2FwLCBuYywgZm4sIHN1bkxlbikgewogIGlmIChwb29sICYmIChwb29sLmNhcCA8IGNhcCB8fCBwb29sLnZjYXAgPCB2Y2FwIHx8IHBvb2wubmMgIT09IG5jIHx8IHBvb2wuZm4gIT09IGZuIHx8IHBvb2wuc3VuTGVuICE9PSBzdW5MZW4pKSB7CiAgICBmb3IgKGNvbnN0IHcgb2YgcG9vbC53b3JrZXJzKSB3LnRlcm1pbmF0ZSgpOwogICAgcG9vbCA9IG51bGw7CiAgfQogIGlmICghcG9vbCkgcG9vbCA9IGNyZWF0ZVBvb2woTWF0aC5tYXgoTWF0aC5jZWlsKGNhcCAqIDEuMjUpLCBDRkcuTUFYX0NFTExTICsgMTAwMCksIE1hdGgubWF4KE1hdGguY2VpbCh2Y2FwICogMS4yNSksIENGRy5NQVhfVklSVVNFUyArIDEwMDApLCBuYywgZm4sIHN1bkxlbik7CiAgcmV0dXJuIHBvb2wuVjsKfQoKLy8gUnVuIGEgcGhhc2Ugb3ZlciBhbGwgY2h1bmtzLCB3aXRoIHRoZSBoZWxwZXJzIGlmIFYgaXMgdGhlIHBvb2wncyBzaGFyZWQgc2V0LgpleHBvcnQgZnVuY3Rpb24gcnVuKFYsIHBoYXNlKSB7CiAgVi5QW1BfQ0hVTktTXSA9IENIVU5LUzsKICBjb25zdCBwID0gcG9vbDsKICAvLyBIZWxwZXJzIGpvaW4gb25seSBvbmNlIGV2ZXJ5IG9uZSBvZiB0aGVtIGlzIHdhaXRpbmcgKHRoZXkgbG9hZCBhc3luY2hyb25vdXNseSk7IHVudGlsIHRoZW4sIG9yIHdpdGhvdXQKICAvLyBzaGFyZWQgbWVtb3J5LCB0aGUgbWFpbiB0aHJlYWQgZG9lcyBhbGwgdGhlIGNodW5rcyBpdHNlbGYgKHdpdGggaWRlbnRpY2FsIHJlc3VsdHMpLgogIGlmICghcCB8fCBWICE9PSBwLlYgfHwgIXAuc2hhcmVkIHx8IHAuaGVscGVycyA9PT0gMCB8fCBBdG9taWNzLmxvYWQocC5jdGwsIENfUkVBRFkpIDwgcC5oZWxwZXJzKSB7CiAgICBmb3IgKGxldCBrID0gMDsgayA8IENIVU5LUzsgaysrKSBydW5DaHVuayhWLCBwaGFzZSwgayk7CiAgICByZXR1cm47CiAgfQogIGNvbnN0IGN0bCA9IHAuY3RsOwogIEF0b21pY3Muc3RvcmUoY3RsLCBDX1BIQVNFLCBQSEFTRVMuaW5kZXhPZihwaGFzZSkpOwogIEF0b21pY3Muc3RvcmUoY3RsLCBDX0NIVU5LUywgQ0hVTktTKTsKICBBdG9taWNzLnN0b3JlKGN0bCwgQ19ORVhULCAwKTsKICBBdG9taWNzLnN0b3JlKGN0bCwgQ19ET05FLCAwKTsKICBBdG9taWNzLmFkZChjdGwsIENfR0VOLCAxKTsKICBBdG9taWNzLm5vdGlmeShjdGwsIENfR0VOKTsKICBmb3IgKGxldCBrOyAoayA9IEF0b21pY3MuYWRkKGN0bCwgQ19ORVhULCAxKSkgPCBDSFVOS1M7KSBydW5DaHVuayhWLCBwaGFzZSwgayk7CiAgLy8gKFRoZSBtYWluIHRocmVhZCBtYXkgbm90IGJsb2NrLCBzbyBpdCBzcGlucyBicmllZmx5IHdoaWxlIHRoZSBsYXN0IGhlbHBlcnMgZmluaXNoIHRoZWlyIGNodW5rcy4pCiAgd2hpbGUgKEF0b21pY3MubG9hZChjdGwsIENfRE9ORSkgPCBwLmhlbHBlcnMpIHsgLyogc3BpbiAqLyB9Cn0KCmV4cG9ydCBmdW5jdGlvbiB0aHJlYWRDb3VudCgpIHsgcmV0dXJuIHBvb2wgPyAocG9vbC5zaGFyZWQgPyBwb29sLmhlbHBlcnMgKyAxIDogMSkgOiBoZWxwZXJDb3VudCgpICsgMTsgfQo=",
 "js/rng.js": "Ly8gU2VlZGVkIFBSTkcgKG11bGJlcnJ5MzIpIHNvIGV2ZXJ5IHdvcmxkIGlzIHJlcHJvZHVjaWJsZSBmcm9tIGl0cyBzZWVkLgpleHBvcnQgY2xhc3MgUk5HIHsKICBjb25zdHJ1Y3RvcihzZWVkKSB7CiAgICB0aGlzLnMgPSAoc2VlZCA+Pj4gMCkgfHwgMTsKICAgIHRoaXMuX3NwYXJlID0gbnVsbDsKICB9CiAgbmV4dCgpIHsKICAgIGxldCB0ID0gKHRoaXMucyA9ICh0aGlzLnMgKyAweDZkMmI3OWY1KSA+Pj4gMCk7CiAgICB0ID0gTWF0aC5pbXVsKHQgXiAodCA+Pj4gMTUpLCB0IHwgMSk7CiAgICB0IF49IHQgKyBNYXRoLmltdWwodCBeICh0ID4+PiA3KSwgdCB8IDYxKTsKICAgIHJldHVybiAoKHQgXiAodCA+Pj4gMTQpKSA+Pj4gMCkgLyA0Mjk0OTY3Mjk2OwogIH0KICByYW5nZShhLCBiKSB7IHJldHVybiBhICsgKGIgLSBhKSAqIHRoaXMubmV4dCgpOyB9CiAgaW50KG4pIHsgcmV0dXJuIE1hdGguZmxvb3IodGhpcy5uZXh0KCkgKiBuKTsgfQogIGNoYW5jZShwKSB7IHJldHVybiB0aGlzLm5leHQoKSA8IHA7IH0KICBnYXVzcygpIHsKICAgIGlmICh0aGlzLl9zcGFyZSAhPT0gbnVsbCkgeyBjb25zdCBzID0gdGhpcy5fc3BhcmU7IHRoaXMuX3NwYXJlID0gbnVsbDsgcmV0dXJuIHM7IH0KICAgIGxldCB1ID0gMCwgdiA9IDA7CiAgICB3aGlsZSAodSA9PT0gMCkgdSA9IHRoaXMubmV4dCgpOwogICAgdiA9IHRoaXMubmV4dCgpOwogICAgY29uc3QgbSA9IE1hdGguc3FydCgtMiAqIE1hdGgubG9nKHUpKTsKICAgIHRoaXMuX3NwYXJlID0gbSAqIE1hdGguc2luKDYuMjgzMTg1MzA3ICogdik7CiAgICByZXR1cm4gbSAqIE1hdGguY29zKDYuMjgzMTg1MzA3ICogdik7CiAgfQp9CgpleHBvcnQgZnVuY3Rpb24gaGFzaFNlcShzZXEpIHsKICBsZXQgaCA9IDIxNjYxMzYyNjE7CiAgZm9yIChsZXQgaSA9IDA7IGkgPCBzZXEubGVuZ3RoOyBpKyspIGggPSBNYXRoLmltdWwoaCBeIChzZXFbaV0gKyAxKSwgMTY3Nzc2MTkpOwogIHJldHVybiBoID4+PiAwOwp9Cg==",
 "js/spatial.js": "Ly8gVW5pZm9ybS1ncmlkIHNwYXRpYWwgaGFzaCByZWJ1aWx0IGVhY2ggdGljayB3aXRoIGEgY291bnRpbmcgc29ydC4KZXhwb3J0IGNsYXNzIFNwYXRpYWxIYXNoIHsKICBjb25zdHJ1Y3RvcihXLCBILCBjcykgewogICAgdGhpcy5jcyA9IGNzOwogICAgdGhpcy5ndyA9IE1hdGguY2VpbChXIC8gY3MpOwogICAgdGhpcy5naCA9IE1hdGguY2VpbChIIC8gY3MpOwogICAgdGhpcy5uYyA9IHRoaXMuZ3cgKiB0aGlzLmdoOwogICAgdGhpcy5zdGFydCA9IG5ldyBJbnQzMkFycmF5KHRoaXMubmMgKyAxKTsKICAgIHRoaXMuZmlsbCA9IG5ldyBJbnQzMkFycmF5KHRoaXMubmMpOwogICAgdGhpcy5pdGVtcyA9IG5ldyBJbnQzMkFycmF5KDEwMjQpOwogICAgdGhpcy5jZWxsT2YgPSBuZXcgSW50MzJBcnJheSgxMDI0KTsKICB9CgogIGNlbGwoeCwgeSkgewogICAgbGV0IGN4ID0gTWF0aC5mbG9vcih4IC8gdGhpcy5jcyksIGN5ID0gTWF0aC5mbG9vcih5IC8gdGhpcy5jcyk7CiAgICBpZiAoY3ggPCAwKSBjeCA9IDA7IGVsc2UgaWYgKGN4ID49IHRoaXMuZ3cpIGN4ID0gdGhpcy5ndyAtIDE7CiAgICBpZiAoY3kgPCAwKSBjeSA9IDA7IGVsc2UgaWYgKGN5ID49IHRoaXMuZ2gpIGN5ID0gdGhpcy5naCAtIDE7CiAgICByZXR1cm4gY3kgKiB0aGlzLmd3ICsgY3g7CiAgfQoKICBidWlsZChuLCB4cywgeXMpIHsKICAgIGlmICh0aGlzLml0ZW1zLmxlbmd0aCA8IG4pIHsKICAgICAgdGhpcy5pdGVtcyA9IG5ldyBJbnQzMkFycmF5KG4gKiAyKTsKICAgICAgdGhpcy5jZWxsT2YgPSBuZXcgSW50MzJBcnJheShuICogMik7CiAgICB9CiAgICBjb25zdCBzdGFydCA9IHRoaXMuc3RhcnQsIGNlbGxPZiA9IHRoaXMuY2VsbE9mOwogICAgc3RhcnQuZmlsbCgwKTsKICAgIGZvciAobGV0IGkgPSAwOyBpIDwgbjsgaSsrKSB7CiAgICAgIGNvbnN0IGMgPSB0aGlzLmNlbGwoeHNbaV0sIHlzW2ldKTsKICAgICAgY2VsbE9mW2ldID0gYzsKICAgICAgc3RhcnRbYyArIDFdKys7CiAgICB9CiAgICBmb3IgKGxldCBjID0gMDsgYyA8IHRoaXMubmM7IGMrKykgc3RhcnRbYyArIDFdICs9IHN0YXJ0W2NdOwogICAgdGhpcy5maWxsLnNldChzdGFydC5zdWJhcnJheSgwLCB0aGlzLm5jKSk7CiAgICBmb3IgKGxldCBpID0gMDsgaSA8IG47IGkrKykgdGhpcy5pdGVtc1t0aGlzLmZpbGxbY2VsbE9mW2ldXSsrXSA9IGk7CiAgfQoKICAvLyBUaGUgZ3JpZCBzcXVhcmVzIG92ZXJsYXBwaW5nIGEgcXVlcnkgYm94LCBmb3IgaG90IGxvb3BzIHRoYXQgd2FsayBzdGFydC9pdGVtcyB0aGVtc2VsdmVzCiAgLy8gKHNhbWUgb3JkZXIgYXMgcXVlcnkoKSwgd2l0aG91dCBhIGNhbGxiYWNrIHBlciBpdGVtKTogc2V0cyB4MCwgeDEsIHkwLCB5MS4KICBib3goeCwgeSwgcikgewogICAgY29uc3QgY3MgPSB0aGlzLmNzOwogICAgbGV0IHgwID0gTWF0aC5mbG9vcigoeCAtIHIpIC8gY3MpLCB4MSA9IE1hdGguZmxvb3IoKHggKyByKSAvIGNzKTsKICAgIGxldCB5MCA9IE1hdGguZmxvb3IoKHkgLSByKSAvIGNzKSwgeTEgPSBNYXRoLmZsb29yKCh5ICsgcikgLyBjcyk7CiAgICBpZiAoeDAgPCAwKSB4MCA9IDA7IGlmICh5MCA8IDApIHkwID0gMDsKICAgIGlmICh4MSA+PSB0aGlzLmd3KSB4MSA9IHRoaXMuZ3cgLSAxOyBpZiAoeTEgPj0gdGhpcy5naCkgeTEgPSB0aGlzLmdoIC0gMTsKICAgIHRoaXMueDAgPSB4MDsgdGhpcy54MSA9IHgxOyB0aGlzLnkwID0geTA7IHRoaXMueTEgPSB5MTsKICB9CgogIC8vIENhbGxzIGZuKGluZGV4KSBmb3IgZXZlcnkgaXRlbSBpbiBjZWxscyBvdmVybGFwcGluZyB0aGUgcXVlcnkgYm94LgogIC8vIElmIGZuIHJldHVybnMgdHJ1ZSB0aGUgcXVlcnkgc3RvcHMgZWFybHkuCiAgcXVlcnkoeCwgeSwgciwgZm4pIHsKICAgIGNvbnN0IGNzID0gdGhpcy5jczsKICAgIGxldCB4MCA9IE1hdGguZmxvb3IoKHggLSByKSAvIGNzKSwgeDEgPSBNYXRoLmZsb29yKCh4ICsgcikgLyBjcyk7CiAgICBsZXQgeTAgPSBNYXRoLmZsb29yKCh5IC0gcikgLyBjcyksIHkxID0gTWF0aC5mbG9vcigoeSArIHIpIC8gY3MpOwogICAgaWYgKHgwIDwgMCkgeDAgPSAwOyBpZiAoeTAgPCAwKSB5MCA9IDA7CiAgICBpZiAoeDEgPj0gdGhpcy5ndykgeDEgPSB0aGlzLmd3IC0gMTsgaWYgKHkxID49IHRoaXMuZ2gpIHkxID0gdGhpcy5naCAtIDE7CiAgICBjb25zdCBzdGFydCA9IHRoaXMuc3RhcnQsIGl0ZW1zID0gdGhpcy5pdGVtczsKICAgIGZvciAobGV0IGN5ID0geTA7IGN5IDw9IHkxOyBjeSsrKSB7CiAgICAgIGZvciAobGV0IGN4ID0geDA7IGN4IDw9IHgxOyBjeCsrKSB7CiAgICAgICAgY29uc3QgYyA9IGN5ICogdGhpcy5ndyArIGN4OwogICAgICAgIGZvciAobGV0IGsgPSBzdGFydFtjXSwgZSA9IHN0YXJ0W2MgKyAxXTsgayA8IGU7IGsrKykgewogICAgICAgICAgaWYgKGZuKGl0ZW1zW2tdKSA9PT0gdHJ1ZSkgcmV0dXJuOwogICAgICAgIH0KICAgICAgfQogICAgfQogIH0KCiAgY291bnRBdCh4LCB5KSB7CiAgICBjb25zdCBjID0gdGhpcy5jZWxsKHgsIHkpOwogICAgcmV0dXJuIHRoaXMuc3RhcnRbYyArIDFdIC0gdGhpcy5zdGFydFtjXTsKICB9Cn0K",
 "js/viruses.js": "Ly8gVmlydXNlcyBhcyBwYXJ0aWNsZXMuIFRoZXkgYmVnaW4gYXMgc2VsZmlzaCBnZW5ldGljIGVsZW1lbnRzIHRoYXQgZXNjYXBlIGZyb20gY2VsbHMsIGRyaWZ0IGFuZAovLyBkZWNheSBpbiB0aGUgd2F0ZXIsIGFuZCBpbmZlY3QgY2VsbHMgd2hvc2Ugc3VyZmFjZSByZWNlcHRvcnMgbWF0Y2ggdGhlaXIgYmluZGluZyBrZXkuIEluc2lkZSwKLy8gdGhleSBoaWphY2sgdGhlIGNlbGwsIHRoZW4gYnVyc3QgaXQgYW5kIHJlbGVhc2UgbXV0YXRlZCBjb3BpZXMuIEhvc3RzIGVzY2FwZSBieSBjaGFuZ2luZyB0aGVpcgovLyByZWNlcHRvcnMgb3IgZXZvbHZpbmcgZGVmZW5jZXMsIHNvIGEgaG9zdOKAk3ZpcnVzIGFybXMgcmFjZSBlbWVyZ2VzLiBPY2Nhc2lvbmFsbHkgYSB2aXJ1cyBjYXJyaWVzCi8vIGEgZ2VuZSBmcm9tIGl0cyBsYXN0IGhvc3QgaW50byBhIG5ldyBvbmUgKHRyYW5zZHVjdGlvbikuCmltcG9ydCB7IENGRyB9IGZyb20gJy4vY29uZmlnLmpzJzsKaW1wb3J0ICogYXMgR24gZnJvbSAnLi9nZW5vbWUuanMnOwppbXBvcnQgKiBhcyBLIGZyb20gJy4va2VybmVscy5qcyc7Cgpjb25zdCBDVVIgPSB7IHg6IDAsIHk6IDAgfTsKCmV4cG9ydCBjbGFzcyBWaXJ1c2VzIHsKICBjb25zdHJ1Y3Rvcih3b3JsZCkgewogICAgdGhpcy53ID0gd29ybGQ7CiAgICB0aGlzLmxpc3QgPSBbXTsKICAgIHRoaXMucGVuZGluZyA9IFtdOwogICAgdGhpcy5uZXh0SWQgPSAxOwogICAgdGhpcy5saW5lYWdlcyA9IDA7CiAgICB0aGlzLnN0YXRzID0geyBpbmZlY3Rpb25zOiAwLCBidXJzdHM6IDAsIHRyYW5zZHVjdGlvbnM6IDAgfTsKICB9CgogIG1ha2UoeCwgeSwgazEsIGsyLCBsaW5lYWdlLCBjYXJnbywgZXVrID0gZmFsc2UpIHsKICAgIGNvbnN0IHYgPSB7IGlkOiB0aGlzLm5leHRJZCsrLCB4LCB5LCB4MDogeCwgeTA6IHksIGsxLCBrMiwgbGluZWFnZSwgY2FyZ28sIGFnZTogMCwgYW5nOiBNYXRoLnJhbmRvbSgpICogNi4yODMgfTsKICAgIGlmIChDRkcuVklSVVNfRE9NQUlOKSB2LmV1ayA9ICEhZXVrOyAvLyBpbmZlY3RzIGV1a2FyeW90ZXMgKHRydWUpIG9yIGJhY3RlcmlhIChmYWxzZSkKICAgIHJldHVybiB2OwogIH0KCiAgc3RlcCgpIHsKICAgIGNvbnN0IHcgPSB0aGlzLncsIEwgPSB3LmxpZmUsIHJuZyA9IHcucm5nOwogICAgLy8gQSBuZXcgdmlydXMgbGluZWFnZSBvY2Nhc2lvbmFsbHkgZXNjYXBlcyBmcm9tIGEgaG9zdCBvbmNlIGxpZmUgaXMgYWJ1bmRhbnQuCiAgICBpZiAoTC5jZWxscy5sZW5ndGggPiBDRkcuVklSVVNfT1JJR0lOX1BPUCAmJiB0aGlzLmxpc3QubGVuZ3RoIDwgNDAgJiYgcm5nLm5leHQoKSA8IENGRy5WSVJVU19PUklHSU5fUCkgewogICAgICBjb25zdCBob3N0ID0gTC5jZWxsc1tybmcuaW50KEwuY2VsbHMubGVuZ3RoKV07CiAgICAgIGlmIChob3N0ICYmICFob3N0LmRlYWQpIHsKICAgICAgICB0aGlzLmxpbmVhZ2VzKys7CiAgICAgICAgY29uc3QgYm9ybiA9IFtdOwogICAgICAgIGZvciAobGV0IGsgPSAwOyBrIDwgNjsgaysrKSB7IGNvbnN0IHYgPSB0aGlzLm1ha2UoaG9zdC54ICsgcm5nLnJhbmdlKC1ob3N0LnIsIGhvc3QuciksIGhvc3QueSArIHJuZy5yYW5nZSgtaG9zdC5yLCBob3N0LnIpLCBob3N0LmdbR24uUkVDMV0sIGhvc3QuZ1tHbi5SRUMyXSwgdGhpcy5saW5lYWdlcywgbnVsbCwgaG9zdC5taXRvKTsgdGhpcy5saXN0LnB1c2godik7IGJvcm4ucHVzaCh2KTsgfQogICAgICAgIHcuZW1pdCgndmVzY2FwZScsIDQ1LCB7IGhvc3QsIHZzOiBib3JuIH0pOwogICAgICAgIHcubWlsZXN0b25lKCd2aXJ1cycsICdUaGUgZmlyc3QgdmlydXM6IGEgc2VsZmlzaCBnZW5ldGljIGVsZW1lbnQgZXNjYXBlcyBpdHMgaG9zdCcsIGhvc3QpOwogICAgICAgIHcubG9nKGBWaXJ1cyBsaW5lYWdlICMke3RoaXMubGluZWFnZXN9IGVtZXJnZXNgKTsKICAgICAgfQogICAgfQogICAgY29uc3Qgb3V0ID0gW10sIGNlbGxzID0gTC5jZWxscywgc3BlYzIgPSAyICogQ0ZHLlZJUlVTX1NQRUMgKiBDRkcuVklSVVNfU1BFQzsKICAgIGNvbnN0IFcgPSBDRkcuVywgSCA9IENGRy5ILCBsaXN0ID0gdGhpcy5saXN0LCBudiA9IGxpc3QubGVuZ3RoOwogICAgLy8gMSkgRHJpZnQgKGFuZCBkZWNheSBpbiBVViksIGluIG9yZGVyOiB0aGlzIHVzZXMgdGhlIHNoYXJlZCByYW5kb20gZ2VuZXJhdG9yLgogICAgZm9yIChjb25zdCB2IG9mIGxpc3QpIHsKICAgICAgdi5hZ2UrKzsKICAgICAgLy8gVWx0cmF2aW9sZXQgbGlnaHQgbmVhciB0aGUgc3VyZmFjZSBkZXN0cm95cyBleHBvc2VkIHBhcnRpY2xlcy4KICAgICAgaWYgKHJuZy5uZXh0KCkgPCBDRkcuVklSVVNfREVDQVkgKiAoMSArIHcuc3VuQXQodi55KSAqIDMpKSB7IHYuZGVhZCA9IHRydWU7IHYubmV4dFdoeSA9ICdmZWxsIGFwYXJ0IChVViBsaWdodCBhbmQgZGVjYXkpJzsgY29udGludWU7IH0KICAgICAgdy5jdXJyZW50KHYueCwgdi55LCBDVVIpOwogICAgICB2LnggKz0gKHJuZy5uZXh0KCkgLSAwLjUpICogQ0ZHLlZJUlVTX0QgKyBDVVIueCAqIDAuODsKICAgICAgdi55ICs9IChybmcubmV4dCgpIC0gMC41KSAqIENGRy5WSVJVU19EICsgQ1VSLnkgKiAwLjg7CiAgICAgIGlmICh2LnggPCAwKSB2LnggPSAtdi54OyBlbHNlIGlmICh2LnggPiBXKSB2LnggPSAyICogVyAtIHYueDsKICAgICAgaWYgKHYueSA8IDApIHYueSA9IC12Lnk7IGVsc2UgaWYgKHYueSA+IEgpIHYueSA9IDIgKiBIIC0gdi55OwogICAgICB2LmFuZyArPSAocm5nLm5leHQoKSAtIDAuNSkgKiAwLjM7CiAgICB9CiAgICAvLyAyKSBXaGljaCBjZWxscyBlYWNoIHZpcnVzIHRvdWNoZXMgKGhhbGYgb2YgdGhlbSBzZWFyY2ggZWFjaCB0aWNrKTogaW4gcGFyYWxsZWwgKGtlcm5lbHMuanMpLgogICAgY29uc3QgViA9IEwuVjsKICAgIGlmIChjZWxscy5sZW5ndGggJiYgViAmJiBudiA8PSBWLnZjYXApIHsKICAgICAgZm9yIChsZXQgayA9IDA7IGsgPCBudjsgaysrKSB7CiAgICAgICAgY29uc3QgdiA9IGxpc3Rba107CiAgICAgICAgVi5WWFtrXSA9IHYueDsgVi5WWVtrXSA9IHYueTsKICAgICAgICBWLlZGTEFHU1trXSA9ICghdi5kZWFkICYmICh2LmlkICsgdy50KSAlIDIgPT09IDAgPyAxIDogMCkgfCAodi5ldWsgPyAyIDogMCk7CiAgICAgIH0KICAgICAgVi5QW0suUF9OVl0gPSBudjsgVi5QW0suUF9ET01BSU5dID0gQ0ZHLlZJUlVTX0RPTUFJTiA/IDEgOiAwOwogICAgICBjb25zdCBSID0gTC5ydW5QaGFzZShWLCAndmlydXMnKTsKICAgICAgLy8gMykgSW5mZWN0aW9ucywgb25lIHZpcnVzIGF0IGEgdGltZSBpbiBvcmRlciAoY2VsbHMgYWxyZWFkeSBpbmZlY3RlZCB0aGlzIHRpY2sgYXJlIHNraXBwZWQpLgogICAgICBmb3IgKGxldCBrID0gMDsgayA8IG52OyBrKyspIHsKICAgICAgICBjb25zdCB2ID0gbGlzdFtrXTsKICAgICAgICBpZiAodi5kZWFkKSBjb250aW51ZTsKICAgICAgICBsZXQgdXNlZCA9IGZhbHNlOwogICAgICAgIGZvciAobGV0IHEgPSBSLlZTVEFSVFtrXSwgcWUgPSBxICsgUi5WQ09VTlRba107IHEgPCBxZTsgcSsrKSB7CiAgICAgICAgICBjb25zdCBjID0gY2VsbHNbUi5WQ1txXV07CiAgICAgICAgICBpZiAoIWMgfHwgYy5kZWFkIHx8IGMuaW5mZWN0ZWQpIGNvbnRpbnVlOwogICAgICAgICAgLy8gQmluZGluZzogdGhlIHZpcnVzJ3Mga2V5IG11c3QgZml0IHRoZSBjZWxsJ3MgcmVjZXB0b3Igc2hhcGUuCiAgICAgICAgICBjb25zdCBkMSA9IGMuZ1tHbi5SRUMxXSAtIHYuazEsIGQyID0gYy5nW0duLlJFQzJdIC0gdi5rMjsKICAgICAgICAgIGNvbnN0IGZpdCA9IE1hdGguZXhwKC0oZDEgKiBkMSArIGQyICogZDIpIC8gc3BlYzIpOwogICAgICAgICAgaWYgKHJuZy5uZXh0KCkgPiBDRkcuVklSVVNfSU5GRUNUICogZml0ICogKDEgLSAwLjkgKiBjLnJlc2lzdCkpIGNvbnRpbnVlOwogICAgICAgICAgdXNlZCA9IHRydWU7CiAgICAgICAgICB2Lm5leHQgPSBjOwogICAgICAgICAgaWYgKHYuY2FyZ28gJiYgcm5nLm5leHQoKSA8IENGRy5UUkFOU0RVQ0UpIHsKICAgICAgICAgICAgdi5uZXh0V2h5ID0gYHNsaXBwZWQgYSBnZW5lIGludG8gY2VsbCAjJHtjLmlkfWA7CiAgICAgICAgICAgIC8vIFRyYW5zZHVjdGlvbjogdGhlIHZpcnVzIGRlbGl2ZXJzIGEgZ2VuZSBmcm9tIGl0cyBsYXN0IGhvc3QgaW5zdGVhZCBvZiB0YWtpbmcgb3Zlci4KICAgICAgICAgICAgYy5nID0gbmV3IEZsb2F0MzJBcnJheShjLmcpOwogICAgICAgICAgICBjLmdbdi5jYXJnby5pXSA9IHYuY2FyZ28udjsKICAgICAgICAgICAgdGhpcy5zdGF0cy50cmFuc2R1Y3Rpb25zKys7CiAgICAgICAgICAgIHcubGlmZS5yZWRlY29kZShjKTsKICAgICAgICAgIH0gZWxzZSB7CiAgICAgICAgICAgIHYubmV4dFdoeSA9IGBpbmplY3RlZCBpdHMgZ2Vub21lIGludG8gY2VsbCAjJHtjLmlkfWA7CiAgICAgICAgICAgIHcuZW1pdCgnaW5qZWN0JywgNDAsIHsgY2VsbDogYywgdiwgeDogdi54LCB5OiB2LnkgfSk7CiAgICAgICAgICAgIGMuaW5mZWN0ZWQgPSB7IHQ6IHcudCArIENGRy5WSVJVU19MQVRFTkNZICsgcm5nLmludChDRkcuVklSVVNfTEFURU5DWSksIGsxOiB2LmsxLCBrMjogdi5rMiwgbGluZWFnZTogdi5saW5lYWdlLCBzaW5jZTogdy50IH07CiAgICAgICAgICAgIHRoaXMuc3RhdHMuaW5mZWN0aW9ucysrOwogICAgICAgICAgfQogICAgICAgICAgYnJlYWs7CiAgICAgICAgfQogICAgICAgIGlmICh1c2VkKSB2LmRlYWQgPSB0cnVlOwogICAgICB9CiAgICB9CiAgICBmb3IgKGNvbnN0IHYgb2YgbGlzdCkgaWYgKCF2LmRlYWQpIG91dC5wdXNoKHYpOwogICAgZm9yIChjb25zdCB2IG9mIHRoaXMucGVuZGluZykgb3V0LnB1c2godik7CiAgICB0aGlzLnBlbmRpbmcgPSBbXTsKICAgIHRoaXMubGlzdCA9IG91dDsKICB9CgogIC8vIFRoZSBob3N0IGJ1cnN0cyBhbmQgcmVsZWFzZXMgYSBuZXcgZ2VuZXJhdGlvbiwgZWFjaCB3aXRoIGEgc2xpZ2h0bHkgbXV0YXRlZCBrZXkuCiAgbHlzZShjKSB7CiAgICBjb25zdCBybmcgPSB0aGlzLncucm5nLCBpbmYgPSBjLmluZmVjdGVkOwogICAgaWYgKCFpbmYpIHJldHVybjsKICAgIGNvbnN0IHJvb20gPSBDRkcuTUFYX1ZJUlVTRVMgLSB0aGlzLmxpc3QubGVuZ3RoIC0gdGhpcy5wZW5kaW5nLmxlbmd0aDsKICAgIGNvbnN0IG4gPSBNYXRoLm1heCgwLCBNYXRoLm1pbihyb29tLCBNYXRoLnJvdW5kKENGRy5WSVJVU19CVVJTVCAqIE1hdGgubWluKDQsIDAuNiArIE1hdGguc3FydChjLm0pKSkpKTsKICAgIGZvciAobGV0IGsgPSAwOyBrIDwgbjsgaysrKSB7CiAgICAgIGNvbnN0IGEgPSBybmcubmV4dCgpICogNi4yODMsIHIgPSBybmcubmV4dCgpICogYy5yOwogICAgICAvLyBFYWNoIHByb2dlbnkgbWF5IGNhcnJ5IGEgcmFuZG9tIGhvc3QgZ2VuZSAobm90IHRoZSBuZXV0cmFsIHBpZ21lbnQgb3IgcmVjZXB0b3JzKS4KICAgICAgbGV0IGNhcmdvID0gbnVsbDsKICAgICAgaWYgKHJuZy5uZXh0KCkgPCAwLjMpIHsKICAgICAgICBsZXQgaSA9IHJuZy5pbnQoR24uTkcpOwogICAgICAgIGlmIChpID09PSBHbi5IVUUgfHwgaSA9PT0gR24uUkVDMSB8fCBpID09PSBHbi5SRUMyKSBpID0gR24uU0laRTsKICAgICAgICBjYXJnbyA9IHsgaSwgdjogYy5nW2ldIH07CiAgICAgIH0KICAgICAgY29uc3QgY2xhbXAgPSAoeCkgPT4gKHggPCAwID8gMCA6IHggPiAxID8gMSA6IHgpOwogICAgICAvLyBQcm9nZW55IGluZmVjdCB0aGUgc2FtZSBraW5kIG9mIGhvc3QsIGV4Y2VwdCBmb3IgdGhlIHJhcmUgaG9zdCBqdW1wIHRvIHRoZSBvdGhlciBkb21haW4uCiAgICAgIGNvbnN0IGV1ayA9IENGRy5WSVJVU19ET01BSU4gJiYgcm5nLm5leHQoKSA8IENGRy5WSVJVU19KVU1QID8gIWMubWl0byA6IGMubWl0bzsKICAgICAgdGhpcy5wZW5kaW5nLnB1c2godGhpcy5tYWtlKGMueCArIE1hdGguY29zKGEpICogciwgYy55ICsgTWF0aC5zaW4oYSkgKiByLCBjbGFtcChpbmYuazEgKyBybmcuZ2F1c3MoKSAqIENGRy5WSVJVU19NVVQpLCBjbGFtcChpbmYuazIgKyBybmcuZ2F1c3MoKSAqIENGRy5WSVJVU19NVVQpLCBpbmYubGluZWFnZSwgY2FyZ28sIGV1aykpOwogICAgfQogICAgdGhpcy5zdGF0cy5idXJzdHMrKzsKICAgIGlmIChuID4gMCkgeyBjLm5leHQgPSB0aGlzLnBlbmRpbmcuYXQoLTEpOyBjLm5leHRXaHkgPSBgYnVyc3Qgb3BlbiwgcmVsZWFzaW5nICR7bn0gbmV3IHZpcnVzZXM7IGZvbGxvd2luZyBvbmVgOyB9CiAgfQp9Cg==",
 "js/world.js": "...",
 "package.json": "eyJ0eXBlIjoibW9kdWxlIn0="
}
for rel, b64 in FILES.items():
    p = os.path.join(SIM, rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "wb") as f:
        f.write(base64.b64decode(b64))
# Smoke test: a short world must reach the first living cell.
r = subprocess.run([NODE, "check-evolution.mjs", "--worker", "1", "--seed", "1", "--ticks", "2500"],
                   cwd=SIM, capture_output=True, text=True, timeout=600)
print(r.stdout.strip()[:300] or r.stderr[-2000:])


In [ ]:
# ---- Run (resumable) ------------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor, as_completed

RESULTS = os.path.join(WORK, f"results-{RUN_TAG}.jsonl")
done = set()
if os.path.exists(RESULTS):
    for line in open(RESULTS):
        try:
            r = json.loads(line)
            if not r.get("error"):
                done.add((r["config"], r["seed"]))
        except json.JSONDecodeError:
            pass
seeds = range(FIRST_SEED, FIRST_SEED + SEEDS_PER_CONFIG)
# Interleave the settings, so a run that stops early still has a fair comparison.
tasks = [(name, s) for s in seeds for name in CONFIGS if (name, s) not in done]
print(f"{len(done)} worlds already done, {len(tasks)} to go")

def run(task):
    name, seed = task
    cmd = [NODE, "check-evolution.mjs", "--worker", "1", "--seed", str(seed), "--ticks", str(TICKS),
           "--cfg", json.dumps(CONFIGS[name]), "--genes", "1"]
    p = subprocess.run(cmd, cwd=SIM, capture_output=True, text=True)
    try:
        r = json.loads(p.stdout.strip().splitlines()[-1])
    except Exception:
        r = {"seed": seed, "error": (p.stderr or "no output")[-1500:]}
    r["config"] = name
    return r

t0 = time.time()
n = len(tasks)
with ThreadPoolExecutor(WORKERS) as ex, open(RESULTS, "a") as out:
    futs = [ex.submit(run, t) for t in tasks]
    for k, f in enumerate(as_completed(futs), 1):
        r = f.result()
        out.write(json.dumps(r) + "\n"); out.flush()
        if r.get("error"):
            print("FAILED", r["config"], r["seed"], r["error"][-300:])
        if k == n or k % max(1, n // 40) == 0:
            el = time.time() - t0
            print(f"{k}/{n} worlds, {el/60:.1f} min elapsed, ~{el / k * (n - k) / 60:.0f} min left (rough)")
print("Done:", RESULTS)


In [ ]:
# ---- Analysis -------------------------------------------------------------------------------------
import pandas as pd, numpy as np
from scipy.stats import fisher_exact, mannwhitneyu
def show(df):
    try:
        display(df)
    except NameError:
        print(df.to_string())

STAGES = [("replicator", "Replicator"), ("protocell", "Protocell"), ("cell", "Living cell"),
          ("translation", "Proteins / genetic code"), ("dna", "DNA genome"), ("photo", "Photosynthesis"),
          ("virus", "Viruses"), ("goe", "Great Oxidation"), ("eukaryote", "Eukaryote"),
          ("sex", "Sexual reproduction"), ("multicellular", "Multicellular"), ("complex", "Complex organism")]
runs = [json.loads(l) for l in open(RESULTS)]
runs = [r for r in runs if not r.get("error") and r["config"] in CONFIGS]
names = list(CONFIGS)
base = names[0]
by = {c: [r for r in runs if r["config"] == c] for c in names}
print({c: len(v) for c, v in by.items()}, "worlds per setting")

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (c - h, c + h)

def holm(ps):
    order = np.argsort(ps); adj = np.empty(len(ps)); m = len(ps); run = 0
    for i, j in enumerate(order):
        run = max(run, min(1, (m - i) * ps[j])); adj[j] = run
    return adj

rows = []
for key, label in STAGES:
    row = {"stage": label}
    for c in names:
        t = [r["ms"][key] for r in by[c] if key in r["ms"]]
        n = len(by[c]); k = len(t); lo, hi = wilson(k, n)
        row[f"{c} %"] = round(100 * k / n, 1) if n else np.nan
        row[f"{c} 95% CI"] = f"{100*lo:.0f}-{100*hi:.0f}"
        row[f"{c} median tick"] = int(np.median(t)) if t else None
    for c in names[1:]:
        kb = sum(key in r["ms"] for r in by[base]); nb = len(by[base])
        kc = sum(key in r["ms"] for r in by[c]); nc = len(by[c])
        row[f"p rate ({c})"] = fisher_exact([[kc, nc - kc], [kb, nb - kb]])[1] if nb and nc else np.nan
        tb = [r["ms"][key] for r in by[base] if key in r["ms"]]; tc = [r["ms"][key] for r in by[c] if key in r["ms"]]
        row[f"p timing ({c})"] = mannwhitneyu(tc, tb).pvalue if len(tb) > 2 and len(tc) > 2 else np.nan
    rows.append(row)
table = pd.DataFrame(rows)
for c in names[1:]:
    ps = table[f"p rate ({c})"].fillna(1).values
    table[f"p rate ({c}) Holm"] = holm(ps)
pd.set_option("display.width", 250)
show(table)
print("p rate: Fisher's exact test on how often the stage happened vs the baseline; p timing: Mann-Whitney U on "
      "when it happened (among worlds that reached it); Holm: corrected for testing 12 stages at once.")


In [ ]:
# ---- Chart: share of worlds reaching each stage, with 95% intervals -----------------------------
import matplotlib.pyplot as plt
x = np.arange(len(STAGES)); w = 0.8 / len(names)
fig, ax = plt.subplots(figsize=(13, 4.5))
for i, c in enumerate(names):
    n = len(by[c]); ks = [sum(k in r["ms"] for r in by[c]) for k, _ in STAGES]
    p = np.array(ks) / max(1, n); ci = np.array([wilson(k, n) for k in ks])
    ax.bar(x + i*w - 0.4 + w/2, 100*p, w, yerr=[np.clip(100*(p - ci[:,0]), 0, None), np.clip(100*(ci[:,1] - p), 0, None)], capsize=3, label=f"{c} (n={n})")
ax.set_xticks(x); ax.set_xticklabels([l for _, l in STAGES], rotation=35, ha="right")
ax.set_ylabel("% of worlds"); ax.set_ylim(0, 105); ax.legend(); ax.set_title(f"Stages reached within {TICKS:,} ticks")
plt.tight_layout(); plt.show()


In [ ]:
# ---- Founders: did the first cells start out different? -------------------------------------------
# A first cell's genome is built from its protocell's best replicator, so a change to early chemistry can
# change every lineage's starting traits. Compare the average founder genome per world.
genes = next((r["geneNames"] for r in runs if r.get("geneNames")), None)
def founder_means(c):
    out = []
    for r in by[c]:
        fs = r.get("founders") or []
        if fs: out.append(np.mean([f["g"] for f in fs], axis=0))
    return np.array(out)
fb = founder_means(base)
for c in names[1:]:
    fc = founder_means(c)
    if len(fb) < 3 or len(fc) < 3:
        print("Not enough founders yet"); continue
    res = []
    for j, g in enumerate(genes):
        p = mannwhitneyu(fc[:, j], fb[:, j]).pvalue
        res.append((g, fb[:, j].mean(), fc[:, j].mean(), fc[:, j].mean() - fb[:, j].mean(), p))
    df = pd.DataFrame(res, columns=["gene", base, c, "difference", "p"]).sort_values("p")
    df["p Holm"] = holm(df["p"].values)
    print(f"Founder genomes, {c} vs {base} (most different first):")
    show(df.head(12))
    for label, f in [("independent origins of life", lambda r: r.get("origins", 0)),
                     ("first cell's tick", lambda r: r["ms"].get("cell", np.nan)),
                     ("founding replicator length (nt)", lambda r: np.mean([x["len"] for x in r.get("founders") or [{"len": np.nan}]]))]:
        a = np.array([f(r) for r in by[base]], float); b = np.array([f(r) for r in by[c]], float)
        a, b = a[~np.isnan(a)], b[~np.isnan(b)]
        p = mannwhitneyu(b, a).pvalue if len(a) > 2 and len(b) > 2 else np.nan
        print(f"  {label}: {base} median {np.median(a):.1f}, {c} median {np.median(b):.1f} (p={p:.3g})")
